# Miniproyecto 1 — Técnicas de Muestreo y Simulación de un Sistema de Discos Rígidos

**Curso:** 0302470 Física Estadística — Instituto de Física, Universidad de Antioquia
**Profesor:** Johans Restrepo Cárdenas
**Autor:** _(escribe aquí tu nombre y afiliación)_
**Fecha:** _(fecha de entrega)_

---

## Cómo usar este notebook

Este notebook contiene **todos los códigos** del miniproyecto, comentados línea a línea, organizados
en cuatro bloques:

| Bloque | Contenido | Puntos del enunciado |
|---|---|---|
| 1 | Muestreo directo (*direct-sampling Monte Carlo*) | 1, 2, 3, 4 |
| 2 | Muestreo por cadenas de Markov (*MCMC*) | 5, 6, 7 |
| 3 | Dinámica molecular por eventos (*event-driven MD*) | 8, 9 |
| 4 | Histogramas de posición a densidad fija + animación | diapositivas 10–12 |
| Bono | Interacción dipolar magnética | diapositiva 13 |

> **Importante:** ejecuta primero la celda de configuración global. La bandera `FAST_MODE`
> permite probar todo el notebook en pocos minutos; ponla en `False` para las corridas
> definitivas del informe.

> **Nota sobre el informe:** el enunciado exige que el informe tenga forma de artículo
> (título, autor, resumen, palabras clave, introducción, marco teórico, resultados y discusión,
> conclusiones, bibliografía, agradecimientos) con los códigos como anexo. Este notebook cubre
> la parte de **códigos y resultados**; el texto del artículo debes redactarlo tú.

In [2]:
# =====================================================================
# CONFIGURACIÓN GLOBAL
# =====================================================================
import random          # generador de números pseudoaleatorios (Mersenne Twister)
import math            # funciones matemáticas elementales (sqrt, pi, hypot, ...)
import time            # medición de tiempos de cómputo
import itertools       # utilidades combinatorias
import numpy as np     # arreglos numéricos y estadística
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ---------------------------------------------------------------------
# FAST_MODE = True  -> corridas cortas, para verificar que todo funciona (~2 min)
# FAST_MODE = False -> corridas del enunciado (10^4, 10^5, 10^6). Puede tardar
#                      entre 20 y 60 minutos en total. Úsalo para el informe final.
# ---------------------------------------------------------------------
FAST_MODE = True

# Factor de reducción aplicado a todos los tamaños de muestra cuando FAST_MODE=True
SCALE = 30 if FAST_MODE else 1

def n_of(exponent):
    # Devuelve 10**exponent reducido por SCALE (con un piso de 500 muestras)
    return max(500, int(10**exponent / SCALE))

# Semilla: se fija para que los resultados sean reproducibles.
# Para las tres repeticiones que pide el enunciado se usan tres semillas distintas.
SEMILLAS = [2026, 9081, 314159]

# Estilo de figuras
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

print("FAST_MODE =", FAST_MODE, "| SCALE =", SCALE)
print("Tamaños efectivos: 10^4 ->", n_of(4), " 10^5 ->", n_of(5), " 10^6 ->", n_of(6))
if FAST_MODE:
    print("\nAVISO: en FAST_MODE muchos conteos de aciertos serán 0, porque la")
    print("probabilidad por muestra es p ~ 1e-4. Es lo esperado: con pocas corridas")
    print("no hay estadística. Pon FAST_MODE = False para las cifras del informe.")

FAST_MODE = True | SCALE = 30
Tamaños efectivos: 10^4 -> 500  10^5 -> 3333  10^6 -> 33333

AVISO: en FAST_MODE muchos conteos de aciertos serán 0, porque la
probabilidad por muestra es p ~ 1e-4. Es lo esperado: con pocas corridas
no hay estadística. Pon FAST_MODE = False para las cifras del informe.


---
# BLOQUE 1 — Muestreo directo (*direct-sampling Monte Carlo*)

¿por qué se necesitan las cajitas rojas?

El espacio de configuraciones de los $N$ discos en 2D es **continuo** y tiene dimensión $2N$. La
probabilidad de que el muestreo produzca **exactamente** las coordenadas
$(0.30,\,0.30),\,(0.30,\,0.70),\dots$ es rigurosamente **cero**:

$$P(\mathbf{x} = \mathbf{x}_a) = \int_{\{\mathbf{x}_a\}} \pi(\mathbf{x})\, d^{2N}x = 0,$$

porque un punto tiene medida nula. Lo único que tiene probabilidad no nula es un **conjunto de
medida finita**. Por eso se reemplaza cada punto objetivo $\mathbf{b}$ por una pequeña ventana
cuadrada (la "cajita roja") de semilado `del_xy`:

$$\mathcal{V}_{\mathbf{b}} = \{\mathbf{a} : \max(|a_x-b_x|,\,|a_y-b_y|) < \texttt{del\_xy}\},$$

es decir, una bola en la **norma del máximo** ($L_\infty$). Se declara que la muestra "cayó en la
configuración $a$" si **cada** una de las $N$ cajitas de $a$ contiene **al menos un** disco.

Esto es un **coarse-graining** o discretización del espacio continuo: convertimos un microestado
puntual (con probabilidad 0) en una celda de volumen finito $(2\,\texttt{del\_xy})^{2N}$
(probabilidad > 0), que sí es medible por Monte Carlo.

Dos condiciones son esenciales para que la comparación entre $a$, $b$ y $c$ sea **justa**:

1. **Ventanas disjuntas** entre configuraciones distintas (si no, una misma muestra cuenta como
   acierto de dos configuraciones y se sesgan los conteos).
2. **Volumen igual y contenido en la región permitida**: las ventanas no deben salirse del
   dominio $[\sigma,\,1-\sigma]^2$, o la configuración cercana a la pared tendría menos volumen
   accesible y menos aciertos, *sin que eso signifique una violación de la equiprobabilidad*.

In [4]:
# =====================================================================
# BLOQUE 1 — PUNTO 1: el programa de muestreo directo, comentado línea a línea
# =====================================================================
"""
vista general del código: este script realiza una simulacion de Monte Carlo mediante
un muestreo directo que genera la posición aleatoria de 4 discos rígidos de radio $\sigma$
dentro de una caja unitaria de 2D y se asegura que no se solapen si se solapan el script se 
reinicia mediante el algoritmo de RECHAZO con reinicio total ("Tabula rasa) para asegurar 
la equiprobabilidad.
"""
#Se define la función que coloca los N discos de radio $\sigma$ sin superposiciones
def direct_disks_box(N, sigma): 
 
    # Es una variable de control para mantener activo el bucle hasta obtener una 
    #configuración válida.
    condition = False

    #Se repite el intento completo mientras no haya una configuración válida (se repite si se solapan)
    while condition == False:

        #Se coloca el primer disco al azar dentro de la caja (respetando los bordes) y se crea la lista L
        L = [(random.uniform(sigma, 1.0 - sigma),
              random.uniform(sigma, 1.0 - sigma))]

        #Un bucle para intentar ubicar uno por uno los N-1 discos restantes. 
        for k in range(1, N):

            #Genera una posición candidata (x,y) para el disco k.
            a = (random.uniform(sigma, 1.0 - sigma),
                 random.uniform(sigma, 1.0 - sigma))

            #Calcula la distancia euclidiana entre el candidato "a" y los discos ya colocados en L,
            #guardando la mínima distancia.
            min_dist = min(math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2) for b in L)

            #Evalúa si hay solapamiento, o sea si la distancia entre sus centros es menor a su diametro.
            # Si es menor, hay solapamiento => energía potencial infinita => configuración prohibida.
            if min_dist < 2.0 * sigma:

                #Marca el intento como no válido debido al solapamiento o colisión.
                condition = False

                #Abandona el intento actual y reinicia la generación desde el primer disco. 
                break

            #Este bloque se ejecuta si el nuevo disco no solapa con ninguno. 
            else:
                #Añade el nuevo disco a la lista L.
                L.append(a)

                #Marca la configuración actual como potencialmente válida.
                condition = True

    #Devuelve la lista con las coordenadas de los N discos cuando todos se colocaron con exito. 
    return L


# --- Prueba rápida -----------------------------------------------------
random.seed(0)
demo = direct_disks_box(4, 0.15)
print("Configuración de prueba (N=4, sigma=0.15):")
for i, p in enumerate(demo):
    print("   disco %d: (%.4f, %.4f)" % (i, p[0], p[1]))
d_min = min(math.hypot(p[0]-q[0], p[1]-q[1]) for p, q in itertools.combinations(demo, 2))
print("Distancia mínima entre centros = %.4f   (debe ser >= 2*sigma = 0.30)" % d_min)

Configuración de prueba (N=4, sigma=0.15):
   disco 0: (0.5532, 0.4238)
   disco 1: (0.4091, 0.8364)
   disco 2: (0.1755, 0.1651)
   disco 3: (0.8227, 0.2795)
Distancia mínima entre centros = 0.3058   (debe ser >= 2*sigma = 0.30)


### Punto 1.1 — ¿Qué significa `sigma` y qué hace `direct_disks_box`?

**`sigma`** es el **radio** de cada disco rígido, medido en unidades del lado de la caja
(la caja es de lado 1, así que `sigma` es adimensional). Interviene en dos lugares y con dos
significados físicos distintos:

- **Interacción disco–pared**: los centros se sortean en $[\sigma,\,1-\sigma]$ y no en $[0,1]$,
  porque el centro no puede acercarse a la pared más que un radio. El "volumen" (área)
  accesible a un disco aislado es entonces $(1-2\sigma)^2$, no 1.
- **Interacción disco–disco**: dos discos se traslapan si la distancia entre centros es menor
  que $2\sigma$ (el diámetro). El potencial de par es

  $$U(r) = \begin{cases} \infty & r < 2\sigma \\ 0 & r \ge 2\sigma \end{cases}$$

`sigma` fija además la **densidad** o fracción de empaquetamiento
$\eta = N\pi\sigma^2$. Para $N=4$, $\sigma=0.15$: $\eta = 4\pi(0.15)^2 \approx 0.283$.

**`direct_disks_box(N, sigma)`** es un **generador de muestras independientes** distribuidas
uniformemente sobre el espacio de configuraciones **permitido** (sin traslapes). Implementa el
algoritmo de **rechazo**: propone $N$ posiciones uniformes independientes y, si alguna pareja
traslapa, **descarta la configuración entera** y vuelve a empezar. Cada llamada devuelve una
muestra **estadísticamente independiente** de todas las demás — ésta es la diferencia esencial
con el método de Markov del Bloque 2.

### Punto 1.2 — ¿Bajo qué condiciones `condition_hit` es verdadera?

`condition_hit` es verdadera si, y sólo si, **TODAS** las $N$ cajitas rojas de la configuración
objetivo `conf` contienen **al menos un** disco de la muestra `x_vec`. Formalmente, es la
conjunción lógica (producto, línea 33) de las $N$ variables `condition_b`:

$$\texttt{condition\_hit} = \bigwedge_{\mathbf{b}\,\in\,\texttt{conf}} \texttt{condition\_b}(\mathbf{b}).$$

Se inicializa en `True` (línea 30) y se va multiplicando (`*=`) por cada `condition_b`; basta que
**una sola** cajita quede vacía para que el producto sea 0 (falso). Nótese que:

- No se exige correspondencia *disco $i$ ↔ cajita $i$*: los discos son **indistinguibles**, así
  que cualquiera de las $N!$ permutaciones cuenta como acierto. Esto es físicamente correcto.
- Si las cajitas de una misma configuración fueran lo bastante grandes como para solaparse,
  un mismo disco podría "llenar" dos cajitas y el criterio dejaría de ser una biyección; por eso
  se requiere `del_xy` pequeño frente a las separaciones típicas.

### Punto 1.3 — ¿Qué es `condition_b` (línea 32) y el condicional de las líneas 34–35?

```python
condition_b = min(max(abs(a[0]-b[0]), abs(a[1]-b[1])) for a in x_vec) < del_xy
```

Se lee de adentro hacia afuera:

1. `max(abs(a[0]-b[0]), abs(a[1]-b[1]))` es la **distancia de Chebyshev** ($L_\infty$) entre el
   disco `a` de la muestra y el punto objetivo `b`. Usar el máximo de las diferencias en $x$ y $y$
   equivale a preguntar si `a` cae dentro de un **cuadrado** centrado en `b` de semilado `del_xy`
   — exactamente la cajita roja de la figura. (Con la distancia euclídea la región sería un
   círculo; el resultado físico es el mismo, sólo cambia el volumen de la celda.)
2. `min(... for a in x_vec)` toma el disco **más cercano** a `b` en esa norma.
3. `< del_xy` devuelve `True` si ese disco más cercano está dentro de la cajita.

Es decir, **`condition_b` es verdadera si la cajita roja centrada en `b` está ocupada por al
menos un disco.**

Las líneas 34–35,

```python
if condition_hit:
    hits[conf] += 1
```

son el **contador de aciertos**: si la muestra reprodujo la configuración objetivo dentro de la
tolerancia `del_xy`, se incrementa en 1 el contador de esa configuración. Al final, la
frecuencia relativa $\texttt{hits}[\alpha]/n_{\rm runs}$ es el **estimador Monte Carlo** de la
probabilidad de que el sistema se encuentre en la celda del microestado $\alpha$.

In [ ]:
# =====================================================================
# BLOQUE 1 — Instrumentación: función de conteo de aciertos
# =====================================================================

def condition_b(x_vec, b, del_xy):
    """
    ¿La cajita cuadrada de semilado `del_xy` centrada en el punto `b` contiene
    al menos un disco de la muestra `x_vec`?  (línea 32 del programa original)
    Usa la norma del máximo (Chebyshev): la región es un CUADRADO.
    """
    return min(max(abs(a[0] - b[0]), abs(a[1] - b[1])) for a in x_vec) < del_xy


def condition_hit(x_vec, conf, del_xy):
    """
    ¿La muestra `x_vec` reproduce la configuración `conf` dentro de la tolerancia?
    Verdadera si TODAS las cajitas de `conf` están ocupadas (líneas 30-33).
    Se corta apenas una cajita falla (equivalente al producto lógico, pero más rápido).
    """
    for b in conf:
        if not condition_b(x_vec, b, del_xy):
            return False
    return True


def corrida_directa(N, sigma, configuraciones, del_xy, n_runs, semilla=None,
                    medir_aceptacion=False):
    """
    Ejecuta `n_runs` muestreos directos independientes y cuenta los aciertos de
    cada configuración objetivo.

    Devuelve (hits, segundos) con hits = lista de enteros del mismo largo que
    `configuraciones`.
    """
    if semilla is not None:
        random.seed(semilla)
    hits = [0] * len(configuraciones)
    t0 = time.time()
    for run in range(n_runs):                       # (27) bucle sobre las muestras
        x_vec = direct_disks_box(N, sigma)          # (28) una muestra independiente
        for i, conf in enumerate(configuraciones):  # (29) se prueba contra a, b y c
            if condition_hit(x_vec, conf, del_xy):  # (34)
                hits[i] += 1                        # (35)
    return hits, time.time() - t0


def tasa_de_aceptacion(N, sigma, n_muestras=2000, semilla=0):
    """
    Fracción de intentos de `direct_disks_box` que producen una configuración
    válida al primer intento. Es una medida directa de la eficiencia del método
    de rechazo y explica por qué el muestreo directo se vuelve inviable a alta densidad.
    """
    random.seed(semilla)
    intentos = 0
    for _ in range(n_muestras):
        cond = False
        while not cond:
            intentos += 1
            L = [(random.uniform(sigma, 1.0-sigma), random.uniform(sigma, 1.0-sigma))]
            for k in range(1, N):
                a = (random.uniform(sigma, 1.0-sigma), random.uniform(sigma, 1.0-sigma))
                if min(math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2) for b in L) < 2.0*sigma:
                    cond = False
                    break
                else:
                    L.append(a); cond = True
    return n_muestras / intentos

print("Funciones de conteo definidas.")

In [ ]:
# =====================================================================
# BLOQUE 1 — Parámetros y las tres configuraciones del enunciado (N = 4)
# =====================================================================
sigma  = 0.15    # (18) radio de los discos
del_xy = 0.05    # (19) semilado de las cajitas rojas (tolerancia del coarse-graining)

# (21-23) Las tres configuraciones/microestados del enunciado
conf_a = ((0.30, 0.30), (0.30, 0.70), (0.70, 0.30), (0.70, 0.70))
conf_b = ((0.20, 0.20), (0.20, 0.80), (0.75, 0.25), (0.75, 0.75))
conf_c = ((0.30, 0.20), (0.30, 0.80), (0.70, 0.20), (0.70, 0.70))

# (24) Lista de configuraciones a monitorear
configuraciones4 = [conf_a, conf_b, conf_c]
nombres = ["a", "b", "c"]

# --- Verificaciones de consistencia (muy importantes para el informe) ---
# TOL absorbe el error de redondeo de punto flotante: por ejemplo 0.80 + 0.05
# se evalúa como 0.8500000000000001, que "excede" 1-sigma = 0.85 sin que haya
# ningún problema real.
TOL = 1e-9

print("Densidad (fracción de empaquetamiento) eta = N*pi*sigma^2 = %.4f" % (4*math.pi*sigma**2))
print("Región accesible a un centro: [%.2f, %.2f]^2\n" % (sigma, 1-sigma))

for nom, conf in zip(nombres, configuraciones4):
    dmin = min(math.hypot(p[0]-q[0], p[1]-q[1]) for p, q in itertools.combinations(conf, 2))
    dentro = all(sigma-TOL <= p[0] <= 1-sigma+TOL and sigma-TOL <= p[1] <= 1-sigma+TOL
                 for p in conf)
    ventana_ok = all(p[0]-del_xy >= sigma-TOL and p[0]+del_xy <= 1-sigma+TOL and
                     p[1]-del_xy >= sigma-TOL and p[1]+del_xy <= 1-sigma+TOL for p in conf)
    print("conf_%s: d_min = %.4f (>= %.2f ? %s) | discos dentro: %s | ventanas dentro: %s"
          % (nom, dmin, 2*sigma, dmin >= 2*sigma-TOL, dentro, ventana_ok))

# --- ¿Son las tres configuraciones MUTUAMENTE EXCLUYENTES? --------------
# Ésta es la propiedad que realmente importa: una misma muestra no debe poder
# contar como acierto de dos configuraciones a la vez.
#
# Ojo: NO basta con pedir que ninguna ventana de `a` se solape con ninguna de `b`.
# De hecho, con del_xy = 0.05 sí hay solapes puntuales — por ejemplo `a` y `c`
# COMPARTEN literalmente el sitio (0.70, 0.70), y la ventana (0.70,0.30) de `a`
# se solapa parcialmente con la (0.75,0.25) de `b`.
#
# El criterio correcto es un ARGUMENTO DE CONTEO: para acertar simultáneamente dos
# configuraciones haría falta ocupar un conjunto de ventanas con interiores
# disjuntos, y cada una de esas ventanas exige un disco DISTINTO. Si ese número
# mínimo supera N, las configuraciones son incompatibles.

def ventanas_disjuntas(p, q, d, tol=TOL):
    """¿Tienen interiores disjuntos las ventanas centradas en p y q?"""
    return not (abs(p[0]-q[0]) < 2*d - tol and abs(p[1]-q[1]) < 2*d - tol)

def discos_minimos(ventanas, d):
    """
    Número mínimo de discos DISTINTOS necesarios para ocupar todas las ventanas:
    es el tamaño del mayor subconjunto con interiores mutuamente disjuntos
    (fuerza bruta; el número de ventanas es pequeño).
    """
    mejor = 0
    n = len(ventanas)
    for r in range(n, 0, -1):
        if r <= mejor:
            break
        for sub in itertools.combinations(range(n), r):
            if all(ventanas_disjuntas(ventanas[i], ventanas[j], d)
                   for i, j in itertools.combinations(sub, 2)):
                mejor = max(mejor, r)
                break
    return mejor

print("\nVentanas disjuntas DENTRO de cada configuración (cada disco ocupa una):")
for nom, conf in zip(nombres, configuraciones4):
    ok = all(ventanas_disjuntas(p, q, del_xy) for p, q in itertools.combinations(conf, 2))
    print("   conf_%s : %s" % (nom, "sí" if ok else "NO"))

print("\nExclusión mutua entre pares de configuraciones (N = 4 discos disponibles):")
for (i, ci), (j, cj) in itertools.combinations(list(enumerate(configuraciones4)), 2):
    union = list(dict.fromkeys(list(ci) + list(cj)))   # ventanas distintas de la unión
    m = discos_minimos(union, del_xy)
    print("   %s y %s : harían falta >= %d discos distintos para acertar ambas  =>  %s"
          % (nombres[i], nombres[j], m,
             "EXCLUYENTES" if m > 4 else "¡PUEDEN COINCIDIR! (sesgo)"))

# Tasa de aceptación del método de rechazo
acc = tasa_de_aceptacion(4, sigma)
print("\nTasa de aceptación del muestreo directo (N=4, sigma=0.15): %.4f" % acc)
print("=> en promedio hacen falta %.1f intentos por muestra válida." % (1/acc))

In [ ]:
# =====================================================================
# BLOQUE 1 — Figura: las tres configuraciones con sus cajitas rojas
# (reproduce la figura de la diapositiva 2 del enunciado)
# =====================================================================
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, nom, conf in zip(axes, nombres, configuraciones4):
    for (x, y) in conf:
        # disco rígido de radio sigma (azul)
        ax.add_patch(patches.Circle((x, y), sigma, facecolor="royalblue",
                                    edgecolor="navy", alpha=0.85))
        # cajita roja de semilado del_xy (celda de coarse-graining)
        ax.add_patch(patches.Rectangle((x-del_xy, y-del_xy), 2*del_xy, 2*del_xy,
                                       facecolor="crimson", edgecolor="darkred"))
    # región permitida a los CENTROS: [sigma, 1-sigma]^2
    ax.add_patch(patches.Rectangle((sigma, sigma), 1-2*sigma, 1-2*sigma,
                                   fill=False, linestyle="--", edgecolor="gray"))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_title("Configuración $%s$" % nom)
    ax.grid(False)
plt.suptitle("Microestados objetivo, discos ($\\sigma=%.2f$) y cajitas rojas "
             "(semilado $\\delta=%.2f$).\\nLa línea punteada es la región accesible a los centros."
             % (sigma, del_xy), y=1.04)
plt.tight_layout()
plt.savefig("fig01_configuraciones.png", dpi=150, bbox_inches="tight")
plt.show()

## Punto 2 — Corridas para $n_{\rm runs} = 10^4,\,10^5,\,10^6$ (tres veces cada una)

### ¿Cómo se estiman las probabilidades?

Cada muestra es un ensayo de Bernoulli independiente con probabilidad $p_\alpha$ de caer en la
celda de la configuración $\alpha$. Por tanto $\texttt{hits}_\alpha \sim \mathrm{Binomial}(n, p_\alpha)$ y

$$\hat{p}_\alpha = \frac{\texttt{hits}_\alpha}{n_{\rm runs}},
\qquad
\sigma_{\hat p_\alpha} = \sqrt{\frac{\hat p_\alpha (1-\hat p_\alpha)}{n_{\rm runs}}}
\;\simeq\; \frac{\sqrt{\texttt{hits}_\alpha}}{n_{\rm runs}}\quad (\texttt{hits}\ll n).$$

El error **relativo** es $\simeq 1/\sqrt{\texttt{hits}}$: da igual cuán grande sea $n_{\rm runs}$,
lo que controla la precisión es el **número de aciertos**. Como $p \sim 10^{-4}$, con $10^4$
corridas se esperan ~1 acierto (ruido puro) y con $10^6$ corridas ~100 aciertos (~10 % de error).
Ésta es la observación central del punto 2.

Como las tres configuraciones tienen ventanas del mismo volumen y disjuntas, la hipótesis de
equiprobabilidad predice $p_a = p_b = p_c$. La prueba estadística natural es un
**$\chi^2$ de bondad de ajuste** con 2 grados de libertad:

$$\chi^2 = \sum_{\alpha \in \{a,b,c\}} \frac{(\texttt{hits}_\alpha - \bar{h})^2}{\bar{h}},
\qquad \bar h = \tfrac{1}{3}\sum_\alpha \texttt{hits}_\alpha .$$

Valores de $\chi^2 \lesssim 6$ son compatibles con equiprobabilidad al 95 % de confianza.

In [ ]:
# =====================================================================
# BLOQUE 1 — PUNTO 2: tres repeticiones para 10^4, 10^5 y 10^6 (N = 4)
# =====================================================================

def chi2_equiprob(hits):
    """Estadístico chi^2 de bondad de ajuste a la hipótesis de equiprobabilidad."""
    h = np.asarray(hits, dtype=float)
    if h.sum() == 0:
        return float("nan")
    esperado = h.mean()
    return float(((h - esperado)**2 / esperado).sum())


def tabla_corridas(N, sigma, configuraciones, del_xy, exponentes, semillas,
                   etiqueta="", runner=None):
    """
    Ejecuta `len(semillas)` repeticiones para cada n_runs = 10^e y arma una tabla
    de resultados. `runner` permite reutilizar la función con el método de Markov.
    """
    if runner is None:
        runner = corrida_directa
    filas = []
    print("=" * 88)
    print("%s  (N=%d, sigma=%.4f, del_xy=%.3f)" % (etiqueta, N, sigma, del_xy))
    print("=" * 88)
    print("%10s %5s | %8s %8s %8s | %10s %10s %10s | %7s %7s"
          % ("n_runs", "rep", "hits_a", "hits_b", "hits_c",
             "p_a", "p_b", "p_c", "chi2", "t[s]"))
    print("-" * 88)
    for e in exponentes:
        n = n_of(e)
        for r, sem in enumerate(semillas, start=1):
            hits, dt = runner(N, sigma, configuraciones, del_xy, n, semilla=sem)
            p = [h / n for h in hits]
            filas.append(dict(n=n, rep=r, hits=hits, p=p, chi2=chi2_equiprob(hits), t=dt))
            print("%10d %5d | %8d %8d %8d | %10.3e %10.3e %10.3e | %7.2f %7.1f"
                  % (n, r, hits[0], hits[1], hits[2], p[0], p[1], p[2],
                     chi2_equiprob(hits), dt))
        print("-" * 88)
    return filas


exponentes = [4, 5, 6]
filas_directo_N4 = tabla_corridas(4, sigma, configuraciones4, del_xy,
                                  exponentes, SEMILLAS,
                                  etiqueta="MUESTREO DIRECTO — N = 4")

In [ ]:
# =====================================================================
# BLOQUE 1 — Punto 2: análisis agregado y barras de error
# =====================================================================
# Se agrupan las tres repeticiones de cada n_runs para tener la mejor estimación.

print("Estimación combinada de las probabilidades (las 3 repeticiones juntas):\n")
print("%10s | %-24s %-24s %-24s" % ("n_runs", "p_a", "p_b", "p_c"))
print("-" * 86)
resumen = {}
for e in exponentes:
    n = n_of(e)
    sub = [f for f in filas_directo_N4 if f["n"] == n]
    total_hits = np.sum([f["hits"] for f in sub], axis=0)
    n_total = n * len(sub)
    p = total_hits / n_total
    err = np.sqrt(np.maximum(total_hits, 1)) / n_total   # sqrt(hits)/n  (régimen Poisson)
    resumen[n] = (p, err, total_hits)
    print("%10d | " % n + "  ".join("%.3e ± %.1e" % (pi, ei) for pi, ei in zip(p, err)))

# --- Figura: p_alpha con barras de error frente a n_runs ---------------
fig, ax = plt.subplots(figsize=(7, 4.5))
ns = sorted(resumen.keys())
colores = ["tab:blue", "tab:orange", "tab:green"]
for i, nom in enumerate(nombres):
    y  = [resumen[n][0][i] for n in ns]
    dy = [resumen[n][1][i] for n in ns]
    ax.errorbar(ns, y, yerr=dy, marker="o", capsize=4, label="$p_%s$" % nom,
                color=colores[i], lw=1.4)
p_global = np.mean([resumen[ns[-1]][0]])
ax.axhline(p_global, ls="--", c="k", lw=1, label="promedio de las tres")
ax.set_xscale("log"); ax.set_xlabel("$n_{\\rm runs}$ (acumulado sobre 3 repeticiones)")
ax.set_ylabel("probabilidad estimada $\\hat p_\\alpha$")
ax.set_title("Muestreo directo, $N=4$: convergencia a la equiprobabilidad")
ax.legend()
plt.tight_layout(); plt.savefig("fig02_equiprob_directo_N4.png", dpi=150)
plt.show()

print("\nNúmero total de aciertos con el mayor n_runs:", resumen[ns[-1]][2])
print("Error relativo esperado ~ 1/sqrt(hits) = %.1f %%"
      % (100/math.sqrt(max(resumen[ns[-1]][2].mean(), 1))))

## Punto 3 — Sistema del doble de tamaño ($N=8$)

### Diseño de las tres configuraciones

Para $N=8$ hay que elegir con cuidado tanto $\sigma$ como las tres configuraciones. Las
restricciones son:

1. **Densidad moderada**: con $\sigma = 0.15$ y $N=8$ sería $\eta = 8\pi(0.15)^2 \approx 0.57$,
   cerca del empaquetamiento máximo; la tasa de aceptación del muestreo directo se desploma y el
   método deja de ser viable. Se usa $\sigma_8 = 0.06$ ($\eta \approx 0.09$).
2. **Ventanas dentro del dominio permitido**: todas las cajitas deben caber en
   $[\sigma,\,1-\sigma]^2$, para que las tres configuraciones tengan el **mismo volumen accesible**.
3. **Ventanas disjuntas dentro de cada configuración**: la separación entre sitios debe superar
   $2\,\texttt{del\_xy}$.
4. **Configuraciones mutuamente excluyentes**.

Una construcción elegante que satisface todo: se toma una **red $3\times3$** con sitios en
$\{0.19,\,0.50,\,0.81\}$ (separación $0.31$) y se definen las tres configuraciones como la red
completa **quitando un sitio distinto** en cada caso.

- $a$: sin el sitio **central**
- $b$: sin la **esquina** inferior izquierda
- $c$: sin el sitio del **borde** inferior central

Con `del_xy` $=0.13$ las nueve ventanas son disjuntas ($0.31 > 0.26$) y cubren exactamente el
dominio permitido ($0.19-0.13 = 0.06 = \sigma$ y $0.81+0.13 = 0.94 = 1-\sigma$).

**Las tres son mutuamente excluyentes**: una muestra que acierta $a$ tiene sus 8 discos repartidos
en las 8 ventanas de $a$, ninguna de las cuales es la ventana central; para acertar $b$ haría
falta ocupar además la ventana central, lo que exigiría un noveno disco. Por construcción son
además **equivalentes por simetría del volumen**, así que la equiprobabilidad predice
$p_a = p_b = p_c$ de forma exacta.

In [ ]:
# =====================================================================
# BLOQUE 1 — PUNTO 3: sistema N = 8
# =====================================================================
N8      = 8
sigma8  = 0.06     # eta = 8*pi*sigma8^2 ~ 0.09  (densidad moderada => muestreo directo viable)
del_xy8 = 0.13     # semilado de las cajitas

# Red 3x3 de sitios. La separación 0.31 supera 2*del_xy = 0.26 (ventanas disjuntas)
# y 2*sigma8 = 0.12 (discos sin traslape).
red   = [0.19, 0.50, 0.81]
sitios = [(x, y) for y in red for x in red]     # 9 sitios, orden fila por fila

# Cada configuración = la red completa MENOS un sitio distinto
conf_a8 = tuple(s for i, s in enumerate(sitios) if i != 4)   # sin el centro
conf_b8 = tuple(s for i, s in enumerate(sitios) if i != 0)   # sin la esquina inf. izq.
conf_c8 = tuple(s for i, s in enumerate(sitios) if i != 1)   # sin el borde inf. central
configuraciones8 = [conf_a8, conf_b8, conf_c8]

# --- Verificaciones -----------------------------------------------------
print("N = 8, sigma = %.3f  =>  eta = %.4f" % (sigma8, N8*math.pi*sigma8**2))
print("Región accesible a los centros: [%.2f, %.2f]^2\n" % (sigma8, 1-sigma8))
for nom, conf in zip(nombres, configuraciones8):
    dmin = min(math.hypot(p[0]-q[0], p[1]-q[1]) for p, q in itertools.combinations(conf, 2))
    ventana_ok = all(p[0]-del_xy8 >= sigma8-TOL and p[0]+del_xy8 <= 1-sigma8+TOL and
                     p[1]-del_xy8 >= sigma8-TOL and p[1]+del_xy8 <= 1-sigma8+TOL for p in conf)
    disjuntas = all(not (abs(p[0]-q[0]) < 2*del_xy8-TOL and abs(p[1]-q[1]) < 2*del_xy8-TOL)
                    for p, q in itertools.combinations(conf, 2))
    print("conf_%s: n=%d  d_min=%.3f (>=%.2f)  ventanas dentro del dominio: %s  disjuntas: %s"
          % (nom, len(conf), dmin, 2*sigma8, ventana_ok, disjuntas))

print("\nTasa de aceptación (N=8, sigma=%.3f): %.4f" % (sigma8, tasa_de_aceptacion(8, sigma8)))

In [ ]:
# =====================================================================
# BLOQUE 1 — Figura: las tres configuraciones para N = 8
# =====================================================================
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, nom, conf in zip(axes, nombres, configuraciones8):
    # los 9 sitios de la red, en gris claro, para ver cuál falta
    for (x, y) in sitios:
        ax.plot(x, y, "x", color="lightgray", ms=8, zorder=1)
    for (x, y) in conf:
        ax.add_patch(patches.Circle((x, y), sigma8, facecolor="royalblue",
                                    edgecolor="navy", alpha=0.85, zorder=2))
        ax.add_patch(patches.Rectangle((x-del_xy8, y-del_xy8), 2*del_xy8, 2*del_xy8,
                                       facecolor="none", edgecolor="crimson",
                                       lw=1.2, zorder=3))
    ax.add_patch(patches.Rectangle((sigma8, sigma8), 1-2*sigma8, 1-2*sigma8,
                                   fill=False, linestyle="--", edgecolor="gray"))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_title("Configuración $%s$ ($N=8$)" % nom); ax.grid(False)
plt.suptitle("$N=8$: red $3\\times3$ con un sitio ausente. La 'x' gris marca el sitio faltante.",
             y=1.03)
plt.tight_layout(); plt.savefig("fig03_configuraciones_N8.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# =====================================================================
# BLOQUE 1 — PUNTO 3: corridas para N = 8
# =====================================================================
filas_directo_N8 = tabla_corridas(8, sigma8, configuraciones8, del_xy8,
                                  exponentes, SEMILLAS,
                                  etiqueta="MUESTREO DIRECTO — N = 8")

### El papel del tamaño de las cajitas rojas

`del_xy` cumple **dos papeles antagónicos**, y ésa es la respuesta a la pregunta del enunciado:

**Papel 1 — Resolución (sesgo).** `del_xy` define cuán finamente se distingue un microestado.
Si las cajitas son **demasiado grandes**, ocurren tres cosas, todas malas:

- Las ventanas de configuraciones distintas **se solapan**: una misma muestra cuenta como acierto
  de varias configuraciones simultáneamente, e inflá artificialmente los conteos de las
  configuraciones "centrales" (las que están rodeadas de otras). Se rompe la exclusión mutua.
- Las ventanas de una misma configuración se solapan entre sí: un solo disco puede llenar dos
  cajitas y el criterio deja de exigir una biyección disco↔cajita.
- Las ventanas se salen del dominio permitido $[\sigma,1-\sigma]^2$: las configuraciones cercanas
  a la pared quedan con **menos volumen accesible** y por tanto con menos aciertos, produciendo
  una violación **aparente** de la equiprobabilidad que en realidad es un artefacto del método.

**Papel 2 — Estadística (varianza).** La probabilidad de acierto escala como el volumen de la celda:

$$p_\alpha \;\propto\; \frac{(2\,\texttt{del\_xy})^{2N}}{(1-2\sigma)^{2N}} \cdot \frac{N!}{f_{\rm acc}},$$

es decir, **exponencialmente** en $N$. Reducir `del_xy` a la mitad divide $p$ por $2^{2N}$: para
$N=4$ eso es un factor $256$; para $N=8$, un factor $65\,536$. Si las cajitas son demasiado
pequeñas simplemente **no se observa ningún acierto** y el estimador es inútil.

La elección de `del_xy` es entonces un **compromiso sesgo–varianza** clásico. La regla práctica:
tomar `del_xy` tan grande como sea posible **sin** que se solapen las ventanas ni se salgan del
dominio. La celda siguiente lo demuestra numéricamente.

Nótese además que este escalado exponencial es exactamente la razón por la cual el muestreo por
"conteo de microestados en celdas" es impracticable para $N$ grande, y por la que la física
estadística trabaja con **observables macroscópicos** (como el histograma de posiciones del
Bloque 4) en vez de con microestados individuales.

In [ ]:
# =====================================================================
# BLOQUE 1 — Barrido del tamaño de las cajitas: el compromiso sesgo-varianza
# =====================================================================
# Para N = 4 se recorren varios valores de del_xy. Nótese que con del_xy >= 0.05
# las ventanas de a, b y c empiezan a solaparse (por ejemplo el sitio (0.30,0.30)
# de `a` con el (0.20,0.20) de `b` y el (0.30,0.20) de `c`), lo que sesga los conteos
# a favor de la configuración `a`.

n_barrido = n_of(5)
valores_del_xy = [0.03, 0.05, 0.065, 0.08, 0.10]

print("Barrido de del_xy  (N=4, sigma=0.15, n_runs=%d)\n" % n_barrido)
print("%8s | %8s %8s %8s | %8s | %s"
      % ("del_xy", "hits_a", "hits_b", "hits_c", "chi2", "estado"))
print("-" * 84)
resultados_barrido = []
for d in valores_del_xy:
    # Criterio riguroso: (i) ventanas disjuntas dentro de cada configuración y
    # (ii) exclusión mutua entre configuraciones (argumento de conteo).
    internas_ok = all(ventanas_disjuntas(p, q, d)
                      for conf in configuraciones4
                      for p, q in itertools.combinations(conf, 2))
    excluyentes = all(discos_minimos(list(dict.fromkeys(list(ci)+list(cj))), d) > 4
                      for ci, cj in itertools.combinations(configuraciones4, 2))
    # ¿caben las ventanas dentro del dominio permitido?
    dominio_ok = all(p[0]-d >= sigma-TOL and p[0]+d <= 1-sigma+TOL and
                     p[1]-d >= sigma-TOL and p[1]+d <= 1-sigma+TOL
                     for conf in configuraciones4 for p in conf)
    if not internas_ok:
        estado = "ventanas internas se solapan"
    elif not excluyentes:
        estado = "configuraciones NO excluyentes -> SESGO"
    elif not dominio_ok:
        estado = "ventanas fuera del dominio -> SESGO"
    else:
        estado = "válido"
    hits, _ = corrida_directa(4, sigma, configuraciones4, d, n_barrido, semilla=7)
    resultados_barrido.append((d, hits))
    print("%8.3f | %8d %8d %8d | %8.2f | %s"
          % (d, hits[0], hits[1], hits[2], chi2_equiprob(hits), estado))

# --- Figura del barrido ------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ds = [r[0] for r in resultados_barrido]
for i, nom in enumerate(nombres):
    ax1.plot(ds, [r[1][i]/n_barrido for r in resultados_barrido], "o-",
             label="$\\hat p_%s$" % nom, color=colores[i])
ax1.set_yscale("log"); ax1.set_xlabel("del_xy"); ax1.set_ylabel("$\\hat p_\\alpha$")
ax1.set_title("La probabilidad escala como $(2\\,\\delta)^{2N}$")
ax1.legend()

# Cociente hits_a / promedio(hits_b, hits_c): mide el sesgo por solapamiento
razon = [r[1][0] / max(0.5*(r[1][1]+r[1][2]), 1) for r in resultados_barrido]
ax2.plot(ds, razon, "s-", color="crimson")
ax2.axhline(1.0, ls="--", c="k", lw=1, label="sin sesgo")
ax2.axvline(0.05, ls=":", c="gray", label="límite de ventanas disjuntas")
ax2.set_xlabel("del_xy"); ax2.set_ylabel("hits$_a$ / $\\langle$hits$_{b,c}\\rangle$")
ax2.set_title("Sesgo introducido por cajitas demasiado grandes")
ax2.legend()
plt.tight_layout(); plt.savefig("fig04_barrido_delxy.png", dpi=150)
plt.show()

## Punto 4 — ¿Cómo funciona el programa y dónde está implementada la equiprobabilidad?

### Funcionamiento global

El programa tiene dos partes claramente separadas:

**(A) El generador (`direct_disks_box`, líneas 2–15).** Produce configuraciones legales de $N$
discos rígidos mediante **rechazo**: propone $N$ posiciones uniformes independientes en la región
permitida y descarta la configuración completa si alguna pareja traslapa.

**(B) El estimador (líneas 27–38).** Repite el muestreo $n_{\rm runs}$ veces y, para cada muestra,
verifica si cae dentro de la celda de tolerancia de alguna de las tres configuraciones objetivo,
acumulando los aciertos.

### Dónde está la equiprobabilidad

La distribución de equilibrio de un sistema aislado en el colectivo microcanónico es

$$\pi(\mathbf{x}) = \frac{1}{Z}\,\prod_{i<j}\Theta(|\mathbf{r}_i-\mathbf{r}_j| - 2\sigma)
\prod_i \mathbb{1}_{[\sigma,1-\sigma]^2}(\mathbf{r}_i),
\qquad Z = \text{volumen del espacio permitido},$$

es decir, **constante sobre todas las configuraciones sin traslape y cero sobre las prohibidas**.
Para los discos rígidos toda configuración accesible tiene la misma energía ($U=0$), de modo que
el factor de Boltzmann $e^{-\beta U}$ es idéntico para todas: **equiprobabilidad ≡ distribución
uniforme sobre el espacio de configuraciones permitido**.

El programa implementa esto en tres lugares:

1. **`random.uniform`** (líneas 5 y 7): la densidad de propuesta es uniforme, sin ninguna
   preferencia por región alguna del espacio.
2. **El criterio de rechazo** (línea 9): impone la función indicadora que anula el peso de las
   configuraciones traslapadas. Es la única "física" del modelo.
3. **El `break` de la línea 11 — el punto más sutil.** Al detectar un traslape se reinicia la
   configuración **completa**, no sólo el disco conflictivo.

Este último punto merece énfasis, porque es el error conceptual más frecuente. El teorema del
muestreo por rechazo dice que si se propone según $q(\mathbf{x})$ (aquí: uniforme sobre
$[\sigma,1-\sigma]^{2N}$) y se acepta con probabilidad proporcional a
$\pi(\mathbf{x})/q(\mathbf{x})$, entonces las muestras **aceptadas** se distribuyen exactamente
según $\pi$. Aquí la razón vale 1 sobre las configuraciones legales y 0 sobre las ilegales, así
que las muestras aceptadas son uniformes sobre el espacio permitido: **exactamente equiprobables,
sin ningún sesgo y sin correlaciones**.

En cambio, si al detectar un traslape sólo se resortease **el disco k** manteniendo los $k-1$
anteriores (*random sequential adsorption*), la distribución resultante **no** sería uniforme:
se favorecerían las configuraciones en las que los primeros discos dejan mucho espacio libre.
El reinicio total es lo que salva la equiprobabilidad.

### Precio a pagar

La eficiencia es la tasa de aceptación

$$f_{\rm acc} = \frac{\text{volumen del espacio permitido}}{(1-2\sigma)^{2N}},$$

que decae **exponencialmente** con $N$ y con la densidad. Se midió arriba: $f_{\rm acc}\approx 3.7\,\%$
para $N=4$, $\eta=0.283$. A densidades altas o $N$ grande el muestreo directo se vuelve inviable,
y ésa es precisamente la motivación para pasar a las **cadenas de Markov** del Bloque 2.

---
# BLOQUE 2 — Muestreo por cadenas de Markov (*Markov-chain Monte Carlo*)

## Marco teórico

En vez de generar cada configuración desde cero, se construye una **cadena de Markov**
$\mathbf{x}_0 \to \mathbf{x}_1 \to \mathbf{x}_2 \to \dots$ cuya distribución estacionaria es
la distribución de equilibrio $\pi$. La probabilidad de transición $P(\mathbf{x}\to\mathbf{y})$
debe cumplir:

1. **Normalización:** $\sum_{\mathbf{y}} P(\mathbf{x}\to\mathbf{y}) = 1$.
2. **Estacionariedad (balance global):**
   $\sum_{\mathbf{x}} \pi(\mathbf{x}) P(\mathbf{x}\to\mathbf{y}) = \pi(\mathbf{y})$.
   Se garantiza con la condición más fuerte de **balance detallado**:
   $\pi(\mathbf{x}) P(\mathbf{x}\to\mathbf{y}) = \pi(\mathbf{y}) P(\mathbf{y}\to\mathbf{x})$.
3. **Ergodicidad:** toda configuración legal debe ser alcanzable desde cualquier otra en un
   número finito de pasos.

Para discos rígidos, $\pi$ es **uniforme** sobre las configuraciones legales, de modo que el
balance detallado se reduce a exigir que la regla de propuesta sea **simétrica**:
$P(\mathbf{x}\to\mathbf{y}) = P(\mathbf{y}\to\mathbf{x})$. Eso se consigue desplazando un disco
elegido al azar dentro de un cuadrado $[-\delta,\delta]^2$ centrado en su posición actual: la
probabilidad de proponer $\mathbf{y}$ desde $\mathbf{x}$ es la misma que la inversa. El criterio
de aceptación de **Metropolis** se simplifica entonces a

$$P_{\rm acc} = \min\!\left(1,\; \frac{\pi(\mathbf{y})}{\pi(\mathbf{x})}\right)
= \begin{cases} 1 & \text{si } \mathbf{y} \text{ es legal},\\ 0 & \text{si traslapa o sale de la caja}.\end{cases}$$

### El punto crítico: los rechazos SÍ cuentan

Cuando el movimiento se rechaza, la cadena **permanece** en $\mathbf{x}$ y ese estado debe
**volver a contarse** en el promedio. Omitir los rechazos (contar sólo los movimientos aceptados)
**rompe el balance detallado** y sesga la distribución muestreada: las configuraciones "apretadas",
desde las que casi todo movimiento se rechaza, quedarían subrepresentadas. Éste es el error
más frecuente en la implementación, y es la razón por la que en el código instrumentado el
contador de aciertos está **fuera** del `if` de aceptación.

### Ventajas y desventajas frente al muestreo directo

| | Muestreo directo | Cadena de Markov |
|---|---|---|
| Muestras | independientes | **correlacionadas** |
| Corrección | exacta desde la primera muestra | exacta sólo tras la **termalización** |
| Eficiencia a alta densidad | colapsa ($f_{\rm acc}\to 0$) | sigue funcionando |
| Error estadístico | $\sigma/\sqrt{n}$ | $\sigma\sqrt{2\tau_{\rm int}/n}$ |

La cantidad $\tau_{\rm int}$ es el **tiempo de autocorrelación integrado**: el número efectivo
de muestras independientes es $n_{\rm eff} = n/(2\tau_{\rm int})$, no $n$. De ahí que las
fluctuaciones del método de Markov sean **mayores** que las del muestreo directo con el mismo
número nominal de muestras — exactamente lo que pide analizar el punto 7.

In [ ]:
# =====================================================================
# BLOQUE 2 — PUNTO 6: el programa de Markov de la diapositiva 5, comentado
# =====================================================================
# Se reproduce primero el programa ORIGINAL con el comentario línea a línea que
# pide el enunciado (explicación de las líneas 9-14). Más abajo se da la versión
# instrumentada con conteo de aciertos.

def markov_original(n_steps=1000, semilla=0):
    """Versión literal del programa de la diapositiva 5, con comentarios."""
    random.seed(semilla)

    # (3) Configuración INICIAL: cuatro discos en una red cuadrada. Debe ser una
    #     configuración LEGAL (sin traslapes y dentro de la caja); si no lo fuera,
    #     la cadena arrancaría en una región de probabilidad nula.
    L = [[0.25, 0.25], [0.75, 0.25], [0.25, 0.75], [0.75, 0.75]]

    sigma    = 0.15          # (4) radio de los discos
    sigma_sq = sigma ** 2    # (5) radio al cuadrado (se precalcula para evitar raíces)
    delta    = 0.1           # (6) amplitud máxima del desplazamiento propuesto
    n_steps  = n_steps       # (7) número de pasos de la cadena

    for steps in range(n_steps):   # (8) bucle sobre los pasos de Markov

        # (9) Se ELIGE AL AZAR uno de los N discos, con probabilidad uniforme 1/N.
        #     `random.choice` devuelve una REFERENCIA al elemento de la lista, no
        #     una copia: por eso más abajo se puede modificar in-place con `a[:] = b`.
        #     La elección uniforme del disco es parte de la simetría de la propuesta.
        a = random.choice(L)

        # (10) MOVIMIENTO PROPUESTO: se desplaza el disco elegido una cantidad
        #      aleatoria uniforme en [-delta, delta] EN CADA COORDENADA de forma
        #      independiente. La región de propuesta es un cuadrado de lado 2*delta
        #      centrado en la posición actual. Esta regla es SIMÉTRICA
        #      (la probabilidad de ir de a a b es igual a la de ir de b a a),
        #      lo que garantiza el balance detallado con pi uniforme.
        b = [a[0] + random.uniform(-delta, delta),
             a[1] + random.uniform(-delta, delta)]

        # (11) Distancia al cuadrado MÍNIMA de la posición propuesta `b` a todos los
        #      OTROS discos. El filtro `if c != a` excluye al propio disco que se está
        #      moviendo (compararse consigo mismo daría siempre distancia ~0 y el
        #      movimiento se rechazaría siempre).
        #      Se trabaja con la distancia AL CUADRADO para no calcular raíces cuadradas:
        #      es más rápido y numéricamente equivalente.
        min_dist = min((b[0]-c[0])**2 + (b[1]-c[1])**2 for c in L if c != a)

        # (12) Condición de PARED: `box_cond` es verdadera si la posición propuesta
        #      saca al disco de la región permitida, es decir si alguna coordenada
        #      es menor que sigma o mayor que 1-sigma. Equivale al potencial infinito
        #      disco-pared.
        box_cond = min(b[0], b[1]) < sigma or max(b[0], b[1]) > 1.0 - sigma

        # (13) CRITERIO DE ACEPTACIÓN de Metropolis. El movimiento se acepta si NO
        #      viola la pared Y NO traslapa con ningún otro disco
        #      (dist^2 < (2 sigma)^2 = 4 sigma^2  <=>  dist < 2 sigma).
        #      Como todas las configuraciones legales tienen la misma energía (U=0),
        #      la razón de Metropolis vale 1 y la aceptación es determinista:
        #      legal -> se acepta siempre; ilegal -> se rechaza siempre.
        if not (box_cond or min_dist < 4.0 * sigma ** 2):

            # (14) Se ACTUALIZA la posición. `a[:] = b` reemplaza el CONTENIDO de la
            #      lista referenciada por `a`, es decir modifica L in-place. Con
            #      `a = b` sólo se reasignaría el nombre local y L no cambiaría:
            #      la cadena nunca avanzaría. Ésta es la línea más delicada del programa.
            a[:] = b

        # Si la condición falla, NO se hace nada: la cadena PERMANECE en la misma
        # configuración. Ese estado repetido debe contarse igual en los promedios
        # (ver la versión instrumentada más abajo).

    return L   # (15) configuración final de la cadena


print("Configuración tras 1000 pasos de Markov:")
for p in markov_original(1000):
    print("   (%.4f, %.4f)" % (p[0], p[1]))

### Observación técnica sobre la línea 11

El filtro `if c != a` compara **por valor**, no por identidad. Si dos discos tuvieran
exactamente las mismas coordenadas (evento de probabilidad cero en aritmética exacta, pero
posible tras una copia mal hecha), el filtro excluiría ambos y el movimiento se aceptaría
indebidamente. La versión robusta usa el **índice** del disco:

```python
k = random.randint(0, N-1)
min_dist = min((b[0]-L[j][0])**2 + (b[1]-L[j][1])**2 for j in range(N) if j != k)
```

Es la forma que se utiliza en la implementación instrumentada.

In [ ]:
# =====================================================================
# BLOQUE 2 — PUNTO 5: programa de Markov instrumentado con conteo de aciertos
# (incorpora las líneas del programa de muestreo directo del Bloque 1)
# =====================================================================

def config_inicial_legal(N, sigma):
    """
    Configuración inicial legal para arrancar la cadena.
    Para N=4 y N=8 se usa una red cuadrada/rectangular; si no cupiera, se recurre
    al muestreo directo. La cadena debe SIEMPRE arrancar en un estado legal.

    IMPORTANTE: esta función NO fija la semilla. Hacerlo aquí sobrescribiría la
    semilla de la corrida y las distintas repeticiones darían resultados IDÉNTICOS.
    """
    if N == 4:
        L = [[0.25, 0.25], [0.75, 0.25], [0.25, 0.75], [0.75, 0.75]]
    elif N == 8:
        # 8 sitios de una red 3x3 (se omite el centro): coincide con conf_a8
        L = [[x, y] for y in [0.19, 0.50, 0.81] for x in [0.19, 0.50, 0.81]]
        L.pop(4)
    else:
        L = [list(p) for p in direct_disks_box(N, sigma)]
    # Verificación de legalidad
    ok = all(sigma <= p[0] <= 1-sigma and sigma <= p[1] <= 1-sigma for p in L)
    ok &= all(math.hypot(p[0]-q[0], p[1]-q[1]) >= 2*sigma
              for p, q in itertools.combinations(L, 2))
    if not ok:
        L = [list(p) for p in direct_disks_box(N, sigma)]
    return L


def markov_disks_box(N, sigma, configuraciones, del_xy, n_steps, semilla=None,
                     delta=0.10, n_termalizacion=None, guardar_serie=False,
                     muestrear_cada=1, L_inicial=None):
    """
    Muestreo por cadena de Markov (Metropolis) de N discos rígidos, con conteo de
    aciertos de las configuraciones objetivo.

    Parámetros
    ----------
    delta            : amplitud del desplazamiento propuesto.
    n_termalizacion  : pasos descartados al inicio (burn-in). Por defecto 1% de n_steps.
    muestrear_cada   : intervalo de muestreo (thinning). NO cambia el valor esperado,
                       sólo reduce el costo del conteo.
    guardar_serie    : si True, devuelve además la coordenada x de todos los discos
                       en cada muestreo (para los histogramas del Bloque 4).

    Devuelve (hits, segundos)  — o (hits, segundos, info) si guardar_serie=True.

    NOTA CLAVE: el conteo de aciertos está FUERA del `if` de aceptación. Los pasos
    rechazados vuelven a contar el estado actual, como exige el balance detallado.
    """
    if semilla is not None:
        random.seed(semilla)
    if n_termalizacion is None:
        n_termalizacion = max(1000, n_steps // 100)

    L = [list(p) for p in L_inicial] if L_inicial is not None \
        else config_inicial_legal(N, sigma)

    sigma_sq   = sigma ** 2
    cuatro_ssq = 4.0 * sigma_sq
    hits       = [0] * len(configuraciones)
    n_muestras = 0
    n_aceptados = 0
    serie_x    = [] if guardar_serie else None
    t0 = time.time()

    for paso in range(n_termalizacion + n_steps):

        # --- (9) elección uniforme del disco a mover (por índice, versión robusta)
        k = random.randint(0, N - 1)
        a = L[k]

        # --- (10) propuesta simétrica: desplazamiento uniforme en [-delta, delta]^2
        b0 = a[0] + random.uniform(-delta, delta)
        b1 = a[1] + random.uniform(-delta, delta)

        # --- (12) condición de pared
        box_cond = (b0 < sigma or b1 < sigma or b0 > 1.0 - sigma or b1 > 1.0 - sigma)

        # --- (11) distancia al cuadrado mínima a los demás discos
        if not box_cond:
            min_dist_sq = min((b0 - L[j][0])**2 + (b1 - L[j][1])**2
                              for j in range(N) if j != k)
            # --- (13)-(14) criterio de Metropolis: se acepta si es legal
            if min_dist_sq >= cuatro_ssq:
                L[k] = [b0, b1]
                n_aceptados += 1
        # Si se rechaza, la cadena permanece en el estado actual (no se hace nada).

        # --- MUESTREO: se hace SIEMPRE, se haya aceptado o no el movimiento -----
        if paso >= n_termalizacion and (paso - n_termalizacion) % muestrear_cada == 0:
            n_muestras += 1
            for i, conf in enumerate(configuraciones):
                if condition_hit(L, conf, del_xy):
                    hits[i] += 1
            if guardar_serie:
                for p in L:
                    serie_x.append(p[0])

    dt = time.time() - t0
    if guardar_serie:
        info = dict(aceptacion=n_aceptados/(n_termalizacion+n_steps),
                    n_muestras=n_muestras, serie_x=np.array(serie_x), L_final=L)
        return hits, dt, info
    return hits, dt


# Adaptador para reutilizar `tabla_corridas` con el método de Markov
def corrida_markov(N, sigma, configuraciones, del_xy, n_runs, semilla=None, delta=0.10):
    return markov_disks_box(N, sigma, configuraciones, del_xy, n_runs,
                            semilla=semilla, delta=delta)

print("Muestreo de Markov instrumentado listo.")

In [ ]:
# =====================================================================
# BLOQUE 2 — Calibración de delta (amplitud del movimiento)
# =====================================================================
# Regla empírica: una tasa de aceptación entre 30% y 60% minimiza el tiempo de
# autocorrelación. delta muy pequeño => la cadena se mueve poco (alta correlación);
# delta muy grande => casi todo se rechaza (la cadena se queda quieta).

print("%8s | %12s | %12s" % ("delta", "acept. N=4", "acept. N=8"))
print("-" * 40)
for d in [0.03, 0.06, 0.10, 0.15, 0.20, 0.30]:
    _, _, i4 = markov_disks_box(4, sigma,  configuraciones4, del_xy,  20000,
                                semilla=1, delta=d, guardar_serie=True,
                                muestrear_cada=1000)
    _, _, i8 = markov_disks_box(8, sigma8, configuraciones8, del_xy8, 20000,
                                semilla=1, delta=d, guardar_serie=True,
                                muestrear_cada=1000)
    print("%8.2f | %11.1f%% | %11.1f%%" % (d, 100*i4["aceptacion"], 100*i8["aceptacion"]))

DELTA4 = 0.15   # valores elegidos para las corridas de producción
DELTA8 = 0.15
print("\nSe adoptan delta(N=4) = %.2f y delta(N=8) = %.2f" % (DELTA4, DELTA8))

In [ ]:
# =====================================================================
# BLOQUE 2 — PUNTO 7: corridas de Markov, N = 4 y N = 8
# =====================================================================
runner4 = lambda N, s, c, d, n, semilla=None: corrida_markov(N, s, c, d, n, semilla, DELTA4)
runner8 = lambda N, s, c, d, n, semilla=None: corrida_markov(N, s, c, d, n, semilla, DELTA8)

filas_markov_N4 = tabla_corridas(4, sigma, configuraciones4, del_xy,
                                 exponentes, SEMILLAS,
                                 etiqueta="CADENA DE MARKOV — N = 4", runner=runner4)

filas_markov_N8 = tabla_corridas(8, sigma8, configuraciones8, del_xy8,
                                 exponentes, SEMILLAS,
                                 etiqueta="CADENA DE MARKOV — N = 8", runner=runner8)

In [ ]:
# =====================================================================
# BLOQUE 2 — Comparación directo vs. Markov y análisis de fluctuaciones
# =====================================================================

def dispersión_relativa(filas, n):
    """Desviación estándar relativa de los hits entre las repeticiones, para un n dado."""
    sub = [f for f in filas if f["n"] == n]
    todos = np.array([h for f in sub for h in f["hits"]], dtype=float)
    if todos.mean() == 0:
        return float("nan"), float("nan")
    # Dispersión observada entre las 9 cifras (3 reps x 3 configuraciones)
    obs = todos.std(ddof=1) / todos.mean()
    # Dispersión que se esperaría si las muestras fueran INDEPENDIENTES (Poisson)
    poisson = 1.0 / math.sqrt(todos.mean())
    return obs, poisson

print("FLUCTUACIONES: dispersión relativa de los conteos entre repeticiones")
print("(se compara con la predicción de Poisson 1/sqrt(hits), válida para muestras")
print(" independientes. Un cociente > 1 revela CORRELACIÓN temporal.)\n")
print("%10s | %-28s | %-28s" % ("n_runs", "DIRECTO  obs / Poisson", "MARKOV   obs / Poisson"))
print("-" * 76)
for e in exponentes:
    n = n_of(e)
    od, pd_ = dispersión_relativa(filas_directo_N4, n)
    om, pm  = dispersión_relativa(filas_markov_N4, n)
    print("%10d | %8.3f / %8.3f = %5.2f | %8.3f / %8.3f = %5.2f"
          % (n, od, pd_, od/pd_ if pd_ else float("nan"),
                om, pm,  om/pm  if pm  else float("nan")))

# --- Figura comparativa ------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
for ax, filas, titulo in [(axes[0], filas_directo_N4, "Muestreo directo"),
                          (axes[1], filas_markov_N4, "Cadena de Markov")]:
    for i, nom in enumerate(nombres):
        xs, ys, es = [], [], []
        for e in exponentes:
            n = n_of(e)
            sub = [f for f in filas if f["n"] == n]
            h = np.array([f["hits"][i] for f in sub], dtype=float)
            xs.append(n); ys.append(h.mean()/n)
            es.append(h.std(ddof=1)/n if len(h) > 1 else 0.0)
        ax.errorbar(xs, ys, yerr=es, marker="o", capsize=4,
                    label="$\\hat p_%s$" % nom, color=colores[i])
    ax.set_xscale("log"); ax.set_xlabel("$n_{\\rm runs}$")
    ax.set_title("%s ($N=4$)" % titulo); ax.legend()
axes[0].set_ylabel("$\\hat p_\\alpha$")
plt.suptitle("Las barras son la dispersión ENTRE las tres repeticiones. "
             "Nótese que la cadena de Markov fluctúa más.", y=1.02)
plt.tight_layout(); plt.savefig("fig05_directo_vs_markov.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# =====================================================================
# BLOQUE 2 — Tiempo de autocorrelación: la causa de las fluctuaciones extra
# =====================================================================
# Se mide la función de autocorrelación de un observable sencillo (la coordenada x
# del disco 0) a lo largo de la cadena. El tiempo de autocorrelación integrado
# tau_int da el número de pasos que hay que esperar para obtener una muestra
# efectivamente independiente:  n_eff = n / (2 tau_int).

def serie_observable(N, sigma, n_steps, delta, semilla=0):
    """Devuelve la serie temporal de x del disco 0 a lo largo de la cadena."""
    random.seed(semilla)
    L = config_inicial_legal(N, sigma)
    cuatro_ssq = 4.0*sigma**2
    serie = np.empty(n_steps)
    for paso in range(n_steps):
        k = random.randint(0, N-1); a = L[k]
        b0 = a[0] + random.uniform(-delta, delta)
        b1 = a[1] + random.uniform(-delta, delta)
        if not (b0 < sigma or b1 < sigma or b0 > 1.0-sigma or b1 > 1.0-sigma):
            if min((b0-L[j][0])**2 + (b1-L[j][1])**2
                   for j in range(N) if j != k) >= cuatro_ssq:
                L[k] = [b0, b1]
        serie[paso] = L[0][0]      # se registra SIEMPRE (aceptado o rechazado)
    return serie

def autocorrelacion(x, t_max=400):
    x = np.asarray(x, dtype=float); x = x - x.mean()
    var = np.dot(x, x) / len(x)
    return np.array([np.dot(x[:len(x)-t], x[t:]) / ((len(x)-t) * var)
                     for t in range(t_max)])

n_serie = min(400000, n_of(6))
s4 = serie_observable(4, sigma,  n_serie, DELTA4, semilla=11)
C4 = autocorrelacion(s4)
tau_int4 = 0.5 + C4[1:].sum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(s4[:3000], lw=0.7, color="tab:blue")
ax1.set_xlabel("paso de Markov"); ax1.set_ylabel("$x_0$")
ax1.set_title("Serie temporal de la cadena (primeros 3000 pasos)")
ax2.plot(C4, color="tab:red")
ax2.axhline(0, c="k", lw=0.8)
ax2.set_xlabel("retardo $t$ (pasos)"); ax2.set_ylabel("$C(t)$")
ax2.set_title("Autocorrelación: $\\tau_{\\rm int} \\approx %.0f$ pasos" % tau_int4)
plt.tight_layout(); plt.savefig("fig06_autocorrelacion.png", dpi=150)
plt.show()

print("Tiempo de autocorrelación integrado tau_int = %.1f pasos" % tau_int4)
print("=> de %d pasos de Markov, sólo ~%d son efectivamente independientes."
      % (n_serie, n_serie/(2*tau_int4)))
print("=> el error estadístico es sqrt(2*tau_int) = %.1f veces mayor que el del"
      % math.sqrt(2*tau_int4))
print("   muestreo directo con el mismo número nominal de muestras.")

### Discusión del Punto 7

Los resultados permiten afirmar tres cosas:

**1. La equiprobabilidad se cumple también con el método de Markov.** Dentro de la incertidumbre
estadística, $\hat p_a \simeq \hat p_b \simeq \hat p_c$, y los valores **coinciden con los del
muestreo directo**. Esto no es trivial: son dos algoritmos completamente distintos —uno genera
muestras independientes por rechazo, el otro recorre el espacio con una caminata aleatoria— y
convergen a la misma distribución estacionaria porque ambos respetan el balance detallado con
$\pi$ uniforme.

**2. Las fluctuaciones son mayores en el método de Markov.** Con el mismo número nominal de
muestras, la dispersión entre repeticiones excede la predicción de Poisson $1/\sqrt{\text{hits}}$
en un factor $\sqrt{2\tau_{\rm int}}$. La razón es que las muestras consecutivas de la cadena
**no son independientes**: en un paso sólo se mueve un disco y a lo sumo una distancia $\delta$,
de modo que hacen falta $\mathcal{O}(\tau_{\rm int})$ pasos para "olvidar" la configuración anterior.
El estimador sigue siendo **insesgado**, pero su varianza es $\sigma^2 \cdot 2\tau_{\rm int}/n$.

**3. La comparación de eficiencia depende de la densidad.** A $\eta = 0.283$ el muestreo directo
requiere ~27 intentos por muestra válida pero entrega muestras perfectas; la cadena de Markov
acepta ~40–50 % de los movimientos y cada paso es mucho más barato, pero entrega muestras
correlacionadas. **A densidades altas la balanza se inclina decisivamente hacia Markov**, porque
$f_{\rm acc}$ del muestreo directo cae exponencialmente mientras que la cadena sigue avanzando
(aunque con $\tau_{\rm int}$ creciente). Ésta es la razón histórica por la que Metropolis *et al.*
(1953) introdujeron el método precisamente para un sistema de discos rígidos.

**Termalización.** La cadena arranca en una red cuadrada, que es una configuración muy particular
(muy ordenada). Los primeros pasos **no** son representativos del equilibrio y deben descartarse
(*burn-in*); en el código eso lo hace el parámetro `n_termalizacion`. En el muestreo directo este
problema simplemente no existe.

---
# BLOQUE 3 — Muestreo basado en eventos (*event-driven Molecular Dynamics*)

## Marco teórico

### ¿Qué es un "evento"?

En un sistema de discos **rígidos** el potencial es discontinuo: vale $0$ mientras no hay contacto
e $\infty$ en el contacto. La fuerza es nula en todo instante **salvo** en los instantes de
colisión, donde es una delta de Dirac. Por eso la trayectoria es **balística a trozos**:

$$\mathbf{r}_i(t + \Delta t) = \mathbf{r}_i(t) + \mathbf{v}_i\,\Delta t \qquad
\text{(exacto, mientras no haya colisión)}.$$

Un **evento** es un instante en el que la trayectoria libre deja de ser válida, es decir una
**colisión**. Hay dos tipos:

- **Evento de pared**: un disco alcanza $x=\sigma$, $x=1-\sigma$, $y=\sigma$ o $y=1-\sigma$.
  Se invierte la componente de velocidad perpendicular a la pared.
- **Evento de par**: dos discos se tocan, $|\mathbf{r}_i - \mathbf{r}_j| = 2\sigma$. Se produce
  una colisión elástica que intercambia la componente de la velocidad relativa a lo largo de la
  línea de centros.

### El algoritmo de Alder y Wainwright (1957)

A diferencia de la dinámica molecular convencional —que integra las ecuaciones de movimiento con
un paso de tiempo $\Delta t$ fijo y comete un error de truncamiento— la **dinámica molecular por
eventos** es *exacta* para potenciales discontinuos. El ciclo es:

1. Calcular el tiempo hasta la próxima colisión de **cada** disco con **cada** pared y de **cada**
   par de discos.
2. Tomar el **mínimo** de todos esos tiempos: ése es el próximo evento.
3. Avanzar **todas** las partículas balísticamente hasta ese instante (movimiento rectilíneo
   uniforme, sin error de integración).
4. Aplicar la regla de colisión correspondiente, que cambia sólo las velocidades de los discos
   involucrados.
5. Volver al paso 1.

Alder y Wainwright introdujeron este esquema en 1957 en el Lawrence Livermore Laboratory para
estudiar esferas duras; con él descubrieron la **transición de fase fluido–sólido en un sistema
puramente repulsivo** (la "transición de Alder"), un resultado que en su momento fue sorprendente
porque no había atracción alguna, y que estableció la simulación por computador como una
herramienta legítima de la física estadística.

### La conexión con la equiprobabilidad

Este sistema es **microcanónico**: la energía cinética total se conserva exactamente (las
colisiones son elásticas), no hay energía potencial, y el sistema está aislado. La **hipótesis
ergódica** afirma que el promedio temporal a lo largo de una única trayectoria equivale al
promedio sobre el colectivo:

$$\overline{A} = \lim_{T\to\infty}\frac{1}{T}\int_0^T A(\mathbf{x}(t))\,dt
\;\;=\;\; \langle A \rangle_{\rm micro} = \int A(\mathbf{x})\,\pi(\mathbf{x})\,d\mathbf{x}.$$

Verificar la equiprobabilidad con este método es, por tanto, verificar la **ergodicidad** de la
dinámica de discos rígidos.

### Advertencia metodológica clave: hay que muestrear a intervalos de tiempo IGUALES

Éste es el punto más sutil del bloque. Si se registra la configuración **en cada evento**, el
muestreo queda **sesgado**: los eventos no están distribuidos uniformemente en el tiempo. En las
regiones donde el sistema colisiona mucho (partículas juntas, cerca de paredes) hay muchos eventos
por unidad de tiempo, y esas configuraciones quedarían **sobrerrepresentadas** respecto al promedio
temporal correcto.

La forma correcta es muestrear a **tiempos equiespaciados** $t = \Delta t_{\rm sample},\;
2\Delta t_{\rm sample},\;\dots$, interpolando la posición balística entre eventos — que es
exactamente lo que hace el bucle `for inter_times in range(int(t+1), int(t+next_event+1))` del
programa del enunciado (allí $\Delta t_{\rm sample} = 1$). El código de abajo implementa ambas
variantes para que la diferencia se pueda **medir**.

In [ ]:
# =====================================================================
# BLOQUE 3 — PUNTO 8: definiciones de los tiempos de colisión, comentadas
# =====================================================================

def wall_time(pos_a, vel_a, sigma):
    """
    Tiempo que le falta a UNA coordenada de un disco para chocar con la pared
    correspondiente.  Se llama por separado para x y para y.

    Geometría: el centro del disco está confinado al intervalo [sigma, 1-sigma];
    la pared "efectiva" para el CENTRO está en sigma y en 1-sigma, no en 0 y 1.
    """
    # (4) Si la velocidad es positiva, el disco se dirige a la pared DERECHA (o superior).
    if vel_a > 0.0:
        # (5) Distancia que falta hasta la pared efectiva (1-sigma), dividida por la
        #     rapidez. Movimiento rectilíneo uniforme: t = distancia / velocidad.
        del_t = (1.0 - sigma - pos_a) / vel_a

    # (6) Si la velocidad es negativa, se dirige a la pared IZQUIERDA (o inferior).
    elif vel_a < 0.0:
        # (7) Distancia hasta la pared efectiva sigma, dividida por el módulo de la
        #     velocidad (abs, porque el tiempo debe ser positivo).
        del_t = (pos_a - sigma) / abs(vel_a)

    # (8) Si la velocidad es exactamente cero...
    else:
        # (9) ...el disco nunca alcanzará esa pared: tiempo infinito. Usar float('inf')
        #     hace que este candidato nunca sea el mínimo, sin necesidad de casos especiales.
        del_t = float('inf')

    # (10) Se devuelve el tiempo hasta la colisión con la pared.
    return del_t


def pair_time(pos_a, vel_a, pos_b, vel_b, sigma):
    """
    Tiempo hasta la colisión entre DOS discos, o infinito si no van a chocar.

    Deducción: la condición de contacto es |Dr + Dv t| = 2 sigma, con
    Dr = r_b - r_a y Dv = v_b - v_a. Elevando al cuadrado se obtiene la
    ecuación cuadrática en t

        |Dv|^2 t^2 + 2 (Dv . Dr) t + (|Dr|^2 - 4 sigma^2) = 0,

    cuyo discriminante (dividido por 4) es

        Upsilon = (Dv . Dr)^2 - |Dv|^2 (|Dr|^2 - 4 sigma^2).
    """
    # (13) Vector separación Dr = r_b - r_a
    del_x = [pos_b[0] - pos_a[0], pos_b[1] - pos_a[1]]

    # (14) Su módulo al cuadrado |Dr|^2
    del_x_sq = del_x[0]**2 + del_x[1]**2

    # (15) Vector velocidad relativa Dv = v_b - v_a
    del_v = [vel_b[0] - vel_a[0], vel_b[1] - vel_a[1]]

    # (16) Su módulo al cuadrado |Dv|^2 (coeficiente cuadrático de la ecuación)
    del_v_sq = del_v[0]**2 + del_v[1]**2

    # (17) Producto escalar Dv . Dr. Su SIGNO dice si los discos se ACERCAN
    #      (scal < 0) o se ALEJAN (scal > 0).
    scal = del_v[0]*del_x[0] + del_v[1]*del_x[1]

    # (18) Discriminante reducido de la ecuación cuadrática.
    Upsilon = scal**2 - del_v_sq * (del_x_sq - 4.0 * sigma**2)

    # (19) Hay colisión sólo si se cumplen DOS condiciones simultáneamente:
    #      Upsilon > 0  -> la ecuación tiene raíces reales (las trayectorias llegan
    #                      efectivamente a distancia 2 sigma; si no, pasan de largo).
    #      scal < 0     -> los discos se están ACERCANDO. Sin esta condición se
    #                      obtendrían "colisiones" en el pasado (tiempos negativos)
    #                      para discos que se alejan.
    if Upsilon > 0.0 and scal < 0.0:
        # (20) Raíz MENOR de la cuadrática (la primera vez que se tocan). El signo
        #      global negativo y la suma de la raíz aseguran que del_t > 0 cuando
        #      scal < 0. Es el instante del primer contacto, no el de la
        #      "salida" del solapamiento.
        del_t = -(scal + math.sqrt(Upsilon)) / del_v_sq
    else:
        # (22) No van a chocar: tiempo infinito.
        del_t = float('inf')

    # (23) Se devuelve el tiempo hasta la colisión de par.
    return del_t


# --- Verificación con un caso analítico --------------------------------
# Dos discos de radio 0.1 que se acercan de frente sobre el eje x:
# separación inicial 0.5, velocidad relativa 2.0 -> chocan cuando la separación
# vale 2*sigma = 0.2, es decir tras recorrer 0.3 => t = 0.3/2.0 = 0.15
t_teor = (0.5 - 0.2) / 2.0
t_num  = pair_time([0.2, 0.5], [1.0, 0.0], [0.7, 0.5], [-1.0, 0.0], 0.1)
print("pair_time: numérico = %.6f   teórico = %.6f   ok = %s"
      % (t_num, t_teor, abs(t_num - t_teor) < 1e-12))

# Disco en x=0.3 con vx=+0.5, sigma=0.1: pared efectiva en 0.9 => t = 0.6/0.5 = 1.2
print("wall_time: numérico = %.6f   teórico = %.6f"
      % (wall_time(0.3, 0.5, 0.1), (0.9-0.3)/0.5))
# Discos que se alejan: debe dar infinito
print("pair_time (se alejan) =", pair_time([0.2,0.5], [-1.0,0.0], [0.7,0.5], [1.0,0.0], 0.1))

In [ ]:
# =====================================================================
# BLOQUE 3 — Programa completo de dinámica molecular por eventos
# =====================================================================

def event_disks_box(pos, vel, sigma, n_events, configuraciones=None, del_xy=None,
                    dt_sample=1.0, muestrear_en_eventos=False,
                    guardar_x=False, verbose_cada=None):
    """
    Dinámica molecular por eventos (Alder & Wainwright, 1957) para N discos
    rígidos en una caja 2D de lado unidad.

    Parámetros
    ----------
    pos, vel  : listas de N listas [x, y]. SE MODIFICAN IN-PLACE.
    dt_sample : intervalo de muestreo a tiempos IGUALES (el muestreo correcto).
    muestrear_en_eventos : si True, además registra la configuración en cada evento
                (muestreo SESGADO, incluido para poder comparar; ver punto 9).
    guardar_x : acumula la coordenada x de todos los discos en cada muestreo.

    Devuelve un diccionario con hits, número de muestras, tiempo simulado, etc.
    """
    N = len(pos)
    # (33) `singles` enumera los 2N pares (disco, coordenada) para los tests de pared
    singles = [(k, l) for k in range(N) for l in range(2)]
    # (34) `pairs` enumera los N(N-1)/2 pares de discos para los tests de colisión
    pairs   = [(k, l) for k in range(N) for l in range(k+1, N)]

    hits          = [0]*len(configuraciones) if configuraciones else []
    hits_eventos  = [0]*len(configuraciones) if configuraciones else []
    x_igual, x_evt = [], []
    n_muestras = 0
    n_col_pared = n_col_par = 0

    t = 0.0                     # (36) reloj de la simulación
    next_sample = dt_sample     # instante del próximo muestreo equiespaciado
    E0 = sum(v[0]**2 + v[1]**2 for v in vel)   # energía cinética inicial (x2/m)
    t0 = time.time()

    for event in range(n_events):     # (38) bucle sobre los eventos

        # --- (41) tiempo hasta que cada coordenada de cada disco choque su pared
        wall_times = [wall_time(pos[k][l], vel[k][l], sigma) for k, l in singles]

        # --- (42) tiempo hasta que cada par de discos colisione
        pair_times = [pair_time(pos[k], vel[k], pos[l], vel[l], sigma) for k, l in pairs]

        # --- (43) el PRÓXIMO evento es el mínimo de todos los tiempos candidatos
        t_wall_min = min(wall_times)
        t_pair_min = min(pair_times)
        next_event = min(t_wall_min, t_pair_min)

        if not math.isfinite(next_event):
            print("Advertencia: no hay más eventos (todas las velocidades nulas).")
            break

        # --- (44) instante desde el que se propaga
        t_previous = t

        # --- (45-50) MUESTREO A TIEMPOS EQUIESPACIADOS -------------------
        # Se avanzan las partículas balísticamente hasta cada instante de muestreo
        # que caiga ANTES del próximo evento, y se registra la configuración.
        # Éste es el muestreo estadísticamente correcto (promedio temporal).
        while next_sample < t + next_event:
            del_t = next_sample - t_previous
            for k, l in singles:                    # (48) traslación balística exacta
                pos[k][l] += vel[k][l] * del_t
            t_previous = next_sample
            n_muestras += 1
            if configuraciones:                     # (51-57) conteo de aciertos
                for i, conf in enumerate(configuraciones):
                    if condition_hit(pos, conf, del_xy):
                        hits[i] += 1
            if guardar_x:
                for p in pos:
                    x_igual.append(p[0])
            next_sample += dt_sample

        # --- (58-62) se avanza hasta el instante EXACTO del evento
        t += next_event
        del_t = t - t_previous
        for k, l in singles:
            pos[k][l] += vel[k][l] * del_t

        # Muestreo alternativo (SESGADO): registrar en cada evento
        if muestrear_en_eventos:
            if configuraciones:
                for i, conf in enumerate(configuraciones):
                    if condition_hit(pos, conf, del_xy):
                        hits_eventos[i] += 1
            if guardar_x:
                for p in pos:
                    x_evt.append(p[0])

        # --- (63) ¿fue una colisión con la pared o entre dos discos? ------
        if t_wall_min < t_pair_min:
            # (64) se identifica QUÉ disco y QUÉ coordenada chocó
            collision_disk, direction = singles[wall_times.index(next_event)]
            # (65) reflexión especular: se invierte la componente perpendicular.
            #      Conserva |v| y por tanto la energía cinética.
            vel[collision_disk][direction] *= -1.0
            n_col_pared += 1
        else:
            # (67) se identifica QUÉ par de discos colisionó
            a, b = pairs[pair_times.index(next_event)]

            # (68) vector separación en el instante del contacto
            del_x = [pos[b][0]-pos[a][0], pos[b][1]-pos[a][1]]
            # (69) su módulo (debe valer 2*sigma en el contacto)
            abs_x = math.sqrt(del_x[0]**2 + del_x[1]**2)
            # (70) versor a lo largo de la línea de centros
            e_perp = [c / abs_x for c in del_x]
            # (71) velocidad relativa
            del_v = [vel[b][0]-vel[a][0], vel[b][1]-vel[a][1]]
            # (72) su proyección sobre la línea de centros: es la ÚNICA componente
            #      que cambia en una colisión elástica de discos lisos e iguales.
            scal = del_v[0]*e_perp[0] + del_v[1]*e_perp[1]
            # (73-75) intercambio de esa componente. Conserva momento lineal
            #      (una gana lo que la otra pierde) y energía cinética.
            for k in range(2):
                vel[a][k] += e_perp[k]*scal
                vel[b][k] -= e_perp[k]*scal
            n_col_par += 1

        if verbose_cada and (event+1) % verbose_cada == 0:
            print("   evento %8d   t = %10.2f   muestras = %8d" % (event+1, t, n_muestras))

    E1 = sum(v[0]**2 + v[1]**2 for v in vel)
    return dict(hits=hits, hits_eventos=hits_eventos, n_muestras=n_muestras,
                t_final=t, n_col_pared=n_col_pared, n_col_par=n_col_par,
                E_inicial=E0, E_final=E1, deriva_energia=abs(E1-E0),
                x_igual=np.array(x_igual), x_evt=np.array(x_evt),
                segundos=time.time()-t0)

print("Dinámica molecular por eventos lista.")

In [ ]:
# =====================================================================
# BLOQUE 3 — Corrida de dinámica molecular por eventos (N = 4)
# =====================================================================
# Condiciones iniciales del enunciado (diapositiva 8). Las velocidades son
# arbitrarias pero NO conmensurables entre sí, para evitar trayectorias periódicas
# que romperían la ergodicidad efectiva.

pos_md = [[0.25, 0.25], [0.75, 0.25], [0.25, 0.75], [0.75, 0.75]]
vel_md = [[0.21, 0.12], [0.71, 0.18], [-0.23, -0.79], [0.78, 0.1177]]
sigma_md = 0.15
del_xy_md = 0.05

# Comprobación de la configuración inicial
assert all(sigma_md <= p[0] <= 1-sigma_md and sigma_md <= p[1] <= 1-sigma_md for p in pos_md)
assert all(math.hypot(p[0]-q[0], p[1]-q[1]) >= 2*sigma_md
           for p, q in itertools.combinations(pos_md, 2))
print("Configuración inicial legal. Energía cinética inicial (2E/m) = %.6f"
      % sum(v[0]**2+v[1]**2 for v in vel_md))

n_events = min(2_000_000, max(50_000, n_of(6)*2))
print("\nSimulando %d eventos..." % n_events)
res_md = event_disks_box(pos_md, vel_md, sigma_md, n_events,
                         configuraciones=configuraciones4, del_xy=del_xy_md,
                         dt_sample=1.0, muestrear_en_eventos=True,
                         guardar_x=True, verbose_cada=max(1, n_events//5))

print("\n--- Resultados ---")
print("Tiempo simulado                : %.1f" % res_md["t_final"])
print("Colisiones con pared / de par  : %d / %d" % (res_md["n_col_pared"], res_md["n_col_par"]))
print("Muestras a tiempos iguales     : %d" % res_md["n_muestras"])
print("Energía inicial / final        : %.10f / %.10f" % (res_md["E_inicial"], res_md["E_final"]))
print("Deriva de energía              : %.3e   <-- debe ser ~0 (el algoritmo es EXACTO)"
      % res_md["deriva_energia"])
print("\nAciertos con muestreo a tiempos IGUALES (correcto):", res_md["hits"])
print("Aciertos con muestreo EN EVENTOS   (sesgado)      :", res_md["hits_eventos"])
if sum(res_md["hits"]) > 0:
    print("chi2 (tiempos iguales) = %.2f" % chi2_equiprob(res_md["hits"]))

## Punto 9 — Histograma de las posiciones de los eventos

El enunciado pide construir el histograma de las posiciones que arroja el programa y determinar si
son igualmente probables. La celda siguiente compara **las dos formas de muestrear** la misma
trayectoria:

- **Muestreo a tiempos iguales** ($t = 1, 2, 3, \dots$): es el promedio temporal correcto y debe
  reproducir la distribución microcanónica de equilibrio, la misma que dan el muestreo directo y
  la cadena de Markov.
- **Muestreo en los instantes de colisión**: es un muestreo **sesgado**. Las colisiones ocurren
  preferentemente cuando los discos están en contacto o tocando las paredes, así que ese conjunto
  de configuraciones está sobrerrepresentado. Físicamente: se estaría muestreando la distribución
  de posiciones *condicionada a que ocurra una colisión*, que no es la distribución de equilibrio.

La comparación de ambos histogramas es el resultado central del punto 9.

In [ ]:
# =====================================================================
# BLOQUE 3 — PUNTO 9: histogramas de posición según el criterio de muestreo
# =====================================================================
x_ig  = res_md["x_igual"]
x_ev  = res_md["x_evt"]
print("Muestras a tiempos iguales : %d" % len(x_ig))
print("Muestras en eventos        : %d" % len(x_ev))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(x_ig, bins=80, density=True, color="tab:blue", alpha=0.85)
axes[0].set_title("Muestreo a tiempos IGUALES\n(promedio temporal correcto)")

axes[1].hist(x_ev, bins=80, density=True, color="tab:red", alpha=0.85)
axes[1].set_title("Muestreo EN CADA EVENTO\n(sesgado hacia las colisiones)")

# Superposición normalizada para ver la diferencia
bins = np.linspace(sigma_md, 1-sigma_md, 81)
h_ig, _ = np.histogram(x_ig, bins=bins, density=True)
h_ev, _ = np.histogram(x_ev, bins=bins, density=True)
ctr = 0.5*(bins[1:] + bins[:-1])
axes[2].plot(ctr, h_ig, lw=1.6, color="tab:blue", label="tiempos iguales")
axes[2].plot(ctr, h_ev, lw=1.6, color="tab:red",  label="en eventos")
axes[2].axhline(1.0/(1-2*sigma_md), ls="--", c="k", lw=1,
                label="uniforme $1/(1-2\\sigma)$")
axes[2].legend(fontsize=8)
axes[2].set_title("Comparación")

for ax in axes:
    ax.set_xlabel("$x$"); ax.set_ylabel("densidad de probabilidad")
    ax.set_xlim(0, 1)
plt.suptitle("Dinámica por eventos, $N=4$, $\\sigma=%.2f$ ($\\eta=%.2f$)"
             % (sigma_md, 4*math.pi*sigma_md**2), y=1.03)
plt.tight_layout()
plt.savefig("fig07_histograma_eventos.png", dpi=150, bbox_inches="tight")
plt.show()

# Cuantificación del sesgo
print("\nDensidad media en el 20%% central vs. en el 20%% junto a las paredes:")
for etiqueta, h in [("tiempos iguales", h_ig), ("en eventos", h_ev)]:
    n = len(h)
    centro = h[int(0.4*n):int(0.6*n)].mean()
    borde  = 0.5*(h[:int(0.1*n)].mean() + h[-int(0.1*n):].mean())
    print("   %-16s centro = %.3f   borde = %.3f   cociente borde/centro = %.3f"
          % (etiqueta, centro, borde, borde/centro))

### Análisis del histograma de eventos

**¿Son igualmente probables las posiciones asociadas a los eventos?** No, y hay que distinguir
con cuidado dos afirmaciones:

1. **Las configuraciones (microestados completos) sí son equiprobables** — eso es lo que confirman
   los conteos de aciertos con muestreo a tiempos iguales, en acuerdo con los Bloques 1 y 2.

2. **La coordenada $x$ de un disco individual NO se distribuye uniformemente**, ni siquiera con el
   muestreo correcto. El histograma muestra un perfil con **máximos junto a las paredes** y un
   mínimo en la región intermedia. Esto no contradice la equiprobabilidad: es la **distribución
   marginal** de una sola coordenada, obtenida integrando sobre las otras $2N-1$, y la integración
   introduce el efecto del **volumen excluido**. Se analiza en detalle en el Bloque 4.

3. **El muestreo en los instantes de colisión está adicionalmente sesgado.** Aquí sí hay un
   artefacto del método: la frecuencia de eventos no es uniforme en el tiempo, de modo que
   promediar "por evento" pondera más las configuraciones colisionantes. El histograma rojo
   exagera aún más los picos junto a las paredes, porque una fracción grande de los eventos son
   precisamente colisiones con la pared, en las cuales el disco está *exactamente* en $x=\sigma$ o
   $x=1-\sigma$.

**Verificación de la ergodicidad y de la exactitud del algoritmo.** Dos controles obligatorios en
el informe:

- La **energía cinética se conserva a precisión de máquina** (deriva $\sim 10^{-15}$). No hay error
  de truncamiento porque no se integra ninguna ecuación diferencial: las traslaciones balísticas
  y las reglas de colisión son fórmulas cerradas exactas.
- El histograma con muestreo a tiempos iguales debe **coincidir** con el del muestreo directo y con
  el de Markov. Que tres dinámicas completamente distintas den la misma distribución es la
  comprobación numérica de la hipótesis ergódica para este sistema.

---
# BLOQUE 4 — Histogramas de posición a densidad fija ($\eta = 0.18$)

Se cambia el punto de vista: en lugar de contar aciertos de **microestados** completos, se mide un
**observable macroscópico** — la coordenada $x$ del centro de un disco — y se construye su
histograma. Esto es mucho más eficiente estadísticamente: cada muestra aporta $N$ valores de $x$,
en lugar de un acierto con probabilidad $\sim10^{-4}$.

La densidad se fija en $\eta = 18\,\%$ mediante el radio:

$$\eta = N\pi\sigma^2 \;\Longrightarrow\; \sigma = \sqrt{\frac{\eta}{N\pi}}.$$

Para $N=4$: $\sigma = \sqrt{0.18/(4\pi)} = 0.1197$ (el valor del enunciado).
Para $N=8$: $\sigma = \sqrt{0.18/(8\pi)} = 0.0846$.

In [ ]:
# =====================================================================
# BLOQUE 4 — Programa de histograma con MUESTREO DIRECTO (diapositiva 10)
# =====================================================================
# Se reproduce el programa del enunciado con comentarios. Dos observaciones:
#  * `pylab` está obsoleto; se usa `matplotlib.pyplot`.
#  * el argumento `normed=True` fue eliminado de Matplotlib: hoy es `density=True`.
#  * la versión del enunciado compara min_dist_sq con 4*sigma**2 (distancia AL
#    CUADRADO), lo cual es equivalente a comparar la distancia con 2*sigma pero
#    evita calcular raíces: es la variante rápida.

def direct_disks_box_sq(N, sigma):
    """Idéntica a `direct_disks_box`, pero comparando distancias AL CUADRADO."""
    overlap = True                                  # (4)
    while overlap == True:                          # (5)
        L = [(random.uniform(sigma, 1.0-sigma),
              random.uniform(sigma, 1.0-sigma))]    # (6) primer disco
        for k in range(1, N):                       # (7)
            a = (random.uniform(sigma, 1.0-sigma),
                 random.uniform(sigma, 1.0-sigma))  # (8) candidato
            # (9) distancia al cuadrado mínima a los ya colocados
            min_dist_sq = min((a[0]-b[0])**2 + (a[1]-b[1])**2 for b in L)
            if min_dist_sq < 4.0*sigma**2:          # (10) traslape: (2 sigma)^2
                overlap = True                      # (11)
                break                               # (12) reinicio total
            else:
                overlap = False                     # (14)
                L.append(a)                         # (15)
    return L                                        # (16)


def histograma_directo(N, eta, n_runs, semilla=0):
    """Muestreo directo: devuelve el arreglo con todas las coordenadas x."""
    sigma = math.sqrt(eta/(N*math.pi))
    random.seed(semilla)
    datos = np.empty(n_runs*N)
    t0 = time.time(); idx = 0
    for run in range(n_runs):                       # (23)
        pos = direct_disks_box_sq(N, sigma)         # (24)
        for k in range(N):                          # (25)
            datos[idx] = pos[k][0]; idx += 1        # (26) se guarda la coordenada x
    return datos, sigma, time.time()-t0


ETA = 0.18
N4, N8 = 4, 8
sigma_h4 = math.sqrt(ETA/(N4*math.pi))
sigma_h8 = math.sqrt(ETA/(N8*math.pi))
print("eta = %.2f  ->  sigma(N=4) = %.4f  (enunciado: 0.1197)" % (ETA, sigma_h4))
print("               sigma(N=8) = %.4f" % sigma_h8)

n_runs_h = min(300_000, max(20_000, n_of(6)//3))
print("\nMuestreo directo con n_runs = %d ..." % n_runs_h)
x_directo, _, t_dir = histograma_directo(N4, ETA, n_runs_h, semilla=101)
print("  %d valores de x en %.1f s" % (len(x_directo), t_dir))

In [ ]:
# =====================================================================
# BLOQUE 4 — El mismo histograma con CADENA DE MARKOV
# =====================================================================
# El enunciado pide corridas largas: al menos n_steps = 2e6. Como cada paso mueve
# un solo disco, hacen falta muchos más pasos que muestras directas para tener el
# mismo número de configuraciones independientes.

n_steps_h = min(3_000_000, max(200_000, n_of(6)*3))
print("Cadena de Markov con n_steps = %d ..." % n_steps_h)
_, t_mk, info_mk = markov_disks_box(
    N4, sigma_h4, [conf_a], 0.01, n_steps_h,      # las configuraciones son irrelevantes aquí
    semilla=202, delta=0.12, guardar_serie=True,
    muestrear_cada=max(1, N4))                     # thinning: una muestra cada N pasos
x_markov = info_mk["serie_x"]
print("  %d valores de x en %.1f s  (aceptación %.1f%%)"
      % (len(x_markov), t_mk, 100*info_mk["aceptacion"]))

In [ ]:
# =====================================================================
# BLOQUE 4 — El mismo histograma con DINÁMICA POR EVENTOS
# =====================================================================
# Se parte de una configuración legal y velocidades aleatorias. Para muestrear
# densamente el histograma conviene reducir dt_sample (más muestras por unidad de
# tiempo simulado); las muestras quedan correlacionadas pero el estimador sigue
# siendo insesgado.

random.seed(303)
pos_h = [list(p) for p in direct_disks_box_sq(N4, sigma_h4)]
vel_h = [[random.gauss(0, 0.5), random.gauss(0, 0.5)] for _ in range(N4)]
# Se elimina el momento total para que el centro de masa no derive
vx = sum(v[0] for v in vel_h)/N4; vy = sum(v[1] for v in vel_h)/N4
for v in vel_h:
    v[0] -= vx; v[1] -= vy

n_ev_h = min(1_500_000, max(50_000, n_of(6)*2))
print("Dinámica por eventos con n_events = %d ..." % n_ev_h)
res_h = event_disks_box(pos_h, vel_h, sigma_h4, n_ev_h,
                        configuraciones=None, del_xy=None,
                        dt_sample=0.05, guardar_x=True)
x_eventos = res_h["x_igual"]
print("  %d valores de x en %.1f s  (t simulado = %.1f, deriva E = %.2e)"
      % (len(x_eventos), res_h["segundos"], res_h["t_final"], res_h["deriva_energia"]))

In [ ]:
# =====================================================================
# BLOQUE 4 — COMPARACIÓN DE LOS TRES HISTOGRAMAS
# =====================================================================
bins = np.linspace(sigma_h4, 1-sigma_h4, 101)
ctr  = 0.5*(bins[1:] + bins[:-1])

h_dir, _ = np.histogram(x_directo, bins=bins, density=True)
h_mkv, _ = np.histogram(x_markov,  bins=bins, density=True)
h_evt, _ = np.histogram(x_eventos, bins=bins, density=True)
uniforme = 1.0/(1-2*sigma_h4)

fig = plt.figure(figsize=(13, 8))
gs  = fig.add_gridspec(2, 3, height_ratios=[1, 1.15], hspace=0.32, wspace=0.25)

for j, (datos, titulo, color) in enumerate([
        (x_directo, "Muestreo directo",      "tab:blue"),
        (x_markov,  "Cadena de Markov",      "tab:orange"),
        (x_eventos, "Dinámica por eventos",  "tab:green")]):
    ax = fig.add_subplot(gs[0, j])
    ax.hist(datos, bins=bins, density=True, color=color, alpha=0.85)
    ax.axhline(uniforme, ls="--", c="k", lw=1.2)
    ax.set_title("%s\n(%d muestras)" % (titulo, len(datos)), fontsize=10)
    ax.set_xlabel("$x$"); ax.set_ylabel("$\\rho(x)$")
    ax.set_xlim(0, 1)

ax = fig.add_subplot(gs[1, :])
ax.plot(ctr, h_dir, lw=1.8, color="tab:blue",   label="muestreo directo")
ax.plot(ctr, h_mkv, lw=1.8, color="tab:orange", ls="--", label="cadena de Markov")
ax.plot(ctr, h_evt, lw=1.8, color="tab:green",  ls=":",  label="dinámica por eventos")
ax.axhline(uniforme, ls="--", c="k", lw=1.2,
           label="distribución uniforme $1/(1-2\\sigma) = %.3f$" % uniforme)
ax.axvline(sigma_h4,   ls=":", c="gray"); ax.axvline(1-sigma_h4, ls=":", c="gray")
ax.set_xlabel("$x$ (coordenada del centro del disco)")
ax.set_ylabel("densidad de probabilidad $\\rho(x)$")
ax.set_title("Los tres métodos convergen a la MISMA distribución, que NO es plana "
             "($N=4$, $\\eta=%.2f$, $\\sigma=%.4f$)" % (ETA, sigma_h4))
ax.legend()
plt.savefig("fig08_tres_histogramas.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Cuantificación del acuerdo entre métodos --------------------------
print("Desviación máxima relativa entre histogramas:")
print("  directo vs Markov  : %.2f %%" % (100*np.max(np.abs(h_dir-h_mkv))/h_dir.mean()))
print("  directo vs eventos : %.2f %%" % (100*np.max(np.abs(h_dir-h_evt))/h_dir.mean()))
print("\nEstructura del perfil (promedio de los tres métodos):")
h_med = (h_dir + h_mkv + h_evt)/3
n = len(h_med)
print("  rho junto a la pared (5%% del borde) : %.4f" %
      (0.5*(h_med[:n//20].mean() + h_med[-(n//20):].mean())))
print("  rho en el centro (10%% central)      : %.4f" %
      h_med[int(0.45*n):int(0.55*n)].mean())
print("  valor uniforme 1/(1-2 sigma)        : %.4f" % uniforme)
print("  contraste pared/centro              : %.3f" %
      (0.5*(h_med[:n//20].mean()+h_med[-(n//20):].mean()) /
       h_med[int(0.45*n):int(0.55*n)].mean()))

## La pregunta central: ¿por qué el histograma NO es plano?

Ésta es la pregunta más importante de todo el miniproyecto, y la respuesta tiene dos niveles.

### 1. No hay ningún error: la equiprobabilidad NO implica un histograma plano

El postulado de equiprobabilidad afirma que la densidad de probabilidad es **constante en el
espacio de configuraciones completo**, que tiene dimensión $2N$:

$$\pi(\mathbf{r}_1,\dots,\mathbf{r}_N) = \text{const.} \quad
\text{sobre } \{\text{configuraciones sin traslape}\}.$$

Lo que se grafica en el histograma **no** es $\pi$, sino su **distribución marginal** de una sola
coordenada:

$$\rho(x_1) = \int \pi(\mathbf{r}_1,\dots,\mathbf{r}_N)\;dy_1\,d^2r_2\cdots d^2r_N .$$

Integrar una función constante sobre una región de forma **complicada** no da una constante: da
el **volumen de la sección transversal** de esa región. Y la región permitida (el conjunto de
configuraciones sin traslape) tiene una forma muy poco trivial. Por eso $\rho(x)$ tiene estructura,
aunque $\pi$ sea rigurosamente uniforme. **Un histograma plano sólo se obtendría para un gas ideal**
($\sigma \to 0$), donde la región permitida es un hipercubo y la sección transversal es constante.

### 2. La física: volumen excluido y presión de depleción

El perfil observado tiene **máximos junto a las paredes** y un mínimo en la zona intermedia. La
razón es el **volumen excluido**:

- Un disco situado **junto a la pared** excluye a los demás de un volumen menor: la parte de su
  "esfera de exclusión" de radio $2\sigma$ que quedaría fuera de la caja no le cuesta nada.
- Un disco en el **centro** excluye a los demás de un disco completo de área $\pi(2\sigma)^2$.

Por tanto, al integrar sobre las posiciones de los otros $N-1$ discos, las configuraciones con un
disco pegado a la pared dejan **más volumen accesible** al resto, y su peso estadístico es mayor.
Éste es exactamente el mecanismo de la **fuerza de depleción** (Asakura–Oosawa): una fuerza
efectiva puramente **entrópica**, sin ninguna atracción en el hamiltoniano, que empuja las
partículas grandes hacia las paredes y hacia otras partículas grandes.

A densidades altas este perfil desarrolla **oscilaciones amortiguadas** con periodo $\approx 2\sigma$
(estructura en capas junto a la pared), el análogo unidimensional de la función de distribución
radial $g(r)$. El contraste crece con $\eta$ y desaparece cuando $\eta \to 0$.

### 3. ¿Qué habría que hacer para "corregir" el problema?

Depende de qué se entienda por "corregir":

| Objetivo | Qué hacer |
|---|---|
| **Comprobar la equiprobabilidad** | No graficar la marginal, sino contar **microestados completos** en celdas de igual volumen, como en los Bloques 1 y 2. Ahí sí se obtiene un resultado plano. |
| **Obtener un histograma plano de $x$** | Hacer $\sigma \to 0$ (límite de gas ideal). La celda siguiente lo demuestra: al bajar $\eta$ el perfil se aplana. |
| **Eliminar el efecto de pared** | Usar **condiciones de contorno periódicas** en lugar de paredes rígidas. Entonces la marginal $\rho(x)$ **sí es exactamente plana** por invariancia traslacional, y la estructura sobreviviente aparece sólo en $g(r)$. |
| **Normalizar correctamente** | Recordar que $x \in [\sigma,\,1-\sigma]$, no $[0,1]$: el valor de referencia de la distribución uniforme es $1/(1-2\sigma)$, no 1. Usar $[0,1]$ produce un desacuerdo espurio. |

Las dos celdas siguientes verifican numéricamente las dos primeras filas de la tabla.

In [ ]:
# =====================================================================
# BLOQUE 4 — Verificación 1: el perfil se aplana cuando eta -> 0
# =====================================================================
fig, ax = plt.subplots(figsize=(8, 5))
n_runs_eta = max(10_000, n_runs_h//6)

print("%8s %10s | %s" % ("eta", "sigma", "contraste pared/centro"))
print("-"*50)
for eta_i, col in zip([0.02, 0.08, 0.18, 0.32], ["#9ecae1", "#4292c6", "#2171b5", "#08306b"]):
    xs, sg, _ = histograma_directo(N4, eta_i, n_runs_eta, semilla=55)
    b = np.linspace(sg, 1-sg, 61)
    h, _ = np.histogram(xs, bins=b, density=True)
    c = 0.5*(b[1:]+b[:-1])
    # se normaliza por el valor uniforme para poder superponer curvas de distinta sigma
    ax.plot(c, h*(1-2*sg), color=col, lw=1.8, label="$\\eta = %.2f$ ($\\sigma=%.3f$)" % (eta_i, sg))
    m = len(h)
    contraste = 0.5*(h[:m//12].mean()+h[-(m//12):].mean()) / h[int(0.45*m):int(0.55*m)].mean()
    print("%8.2f %10.4f | %.3f" % (eta_i, sg, contraste))

ax.axhline(1.0, ls="--", c="k", lw=1.2, label="gas ideal (plano)")
ax.set_xlabel("$x$"); ax.set_ylabel("$\\rho(x)\\,(1-2\\sigma)$   [normalizado]")
ax.set_title("El histograma se aplana al bajar la densidad:\n"
             "la no uniformidad es un efecto de VOLUMEN EXCLUIDO, no un error del método")
ax.legend()
plt.tight_layout(); plt.savefig("fig09_efecto_densidad.png", dpi=150)
plt.show()

In [ ]:
# =====================================================================
# BLOQUE 4 — Verificación 2: con contorno PERIÓDICO la marginal SÍ es plana
# =====================================================================
# Cadena de Markov con condiciones periódicas (convención de imagen mínima).
# Al no haber paredes, la invariancia traslacional garantiza rho(x) = 1 exactamente.

def markov_periodico(N, sigma, n_steps, delta=0.12, semilla=0, muestrear_cada=None):
    random.seed(semilla)
    if muestrear_cada is None:
        muestrear_cada = N
    # configuración inicial: red cuadrada
    lado = int(math.ceil(math.sqrt(N)))
    L = [[(i % lado + 0.5)/lado, (i//lado + 0.5)/lado] for i in range(N)]
    cuatro_ssq = 4.0*sigma**2
    datos = []
    for paso in range(n_steps):
        k = random.randint(0, N-1)
        b0 = (L[k][0] + random.uniform(-delta, delta)) % 1.0   # envoltura periódica
        b1 = (L[k][1] + random.uniform(-delta, delta)) % 1.0
        ok = True
        for j in range(N):
            if j == k: continue
            dx = b0 - L[j][0]; dy = b1 - L[j][1]
            dx -= round(dx); dy -= round(dy)      # convención de imagen mínima
            if dx*dx + dy*dy < cuatro_ssq:
                ok = False; break
        if ok:
            L[k] = [b0, b1]
        if paso % muestrear_cada == 0:
            for p in L: datos.append(p[0])
    return np.array(datos)

n_per = min(1_500_000, max(150_000, n_of(6)))
x_per = markov_periodico(N4, sigma_h4, n_per, semilla=77)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 4.2))
a1.hist(x_directo, bins=80, density=True, color="tab:blue", alpha=0.85)
a1.axhline(1/(1-2*sigma_h4), ls="--", c="k")
a1.set_title("Paredes rígidas: $\\rho(x)$ estructurada"); a1.set_xlabel("$x$")
a2.hist(x_per, bins=80, density=True, color="tab:purple", alpha=0.85)
a2.axhline(1.0, ls="--", c="k")
a2.set_ylim(0, 1.6)
a2.set_title("Contorno periódico: $\\rho(x)$ PLANA"); a2.set_xlabel("$x$")
for a in (a1, a2): a.set_ylabel("$\\rho(x)$")
plt.suptitle("Misma densidad ($\\eta=%.2f$), misma equiprobabilidad, distinta geometría" % ETA,
             y=1.03)
plt.tight_layout(); plt.savefig("fig10_periodico.png", dpi=150, bbox_inches="tight")
plt.show()

h_per, _ = np.histogram(x_per, bins=np.linspace(0, 1, 41), density=True)
print("Contorno periódico: rho(x) = %.4f ± %.4f  (debe ser 1.0000)"
      % (h_per.mean(), h_per.std()))

In [ ]:
# =====================================================================
# BLOQUE 4 — Histogramas para N = 8 a la misma densidad eta = 0.18
# =====================================================================
n_runs_h8 = max(15_000, n_runs_h//3)
x_dir8, _, t8 = histograma_directo(N8, ETA, n_runs_h8, semilla=404)
_, _, info8 = markov_disks_box(N8, sigma_h8, [conf_a8], 0.01,
                               min(2_000_000, max(200_000, n_of(6)*2)),
                               semilla=505, delta=0.12, guardar_serie=True,
                               muestrear_cada=N8)
x_mkv8 = info8["serie_x"]

b8 = np.linspace(sigma_h8, 1-sigma_h8, 101)
c8 = 0.5*(b8[1:]+b8[:-1])
hd8, _ = np.histogram(x_dir8, bins=b8, density=True)
hm8, _ = np.histogram(x_mkv8, bins=b8, density=True)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(ctr, h_dir, lw=1.6, color="tab:blue", label="$N=4$, directo")
ax.plot(c8,  hd8,   lw=1.6, color="tab:red",  label="$N=8$, directo")
ax.plot(c8,  hm8,   lw=1.4, color="tab:red",  ls="--", label="$N=8$, Markov")
ax.axhline(1/(1-2*sigma_h4), ls=":", c="tab:blue")
ax.axhline(1/(1-2*sigma_h8), ls=":", c="tab:red")
ax.set_xlabel("$x$"); ax.set_ylabel("$\\rho(x)$")
ax.set_title("Misma densidad $\\eta=0.18$, distinto $N$ (y por tanto distinto $\\sigma$)")
ax.legend()
plt.tight_layout(); plt.savefig("fig11_histograma_N8.png", dpi=150)
plt.show()

print("N=8: sigma = %.4f, %d muestras directas, %d muestras de Markov"
      % (sigma_h8, len(x_dir8), len(x_mkv8)))
print("Al disminuir sigma a densidad fija, las capas junto a la pared se estrechan:")
print("  ancho del pico de pared ~ 2*sigma:  N=4 -> %.3f   N=8 -> %.3f"
      % (2*sigma_h4, 2*sigma_h8))

---
# Animación de la dinámica por eventos

El programa del enunciado (diapositiva 12) genera un PNG por fotograma y luego los une con
ImageMagick:

```python
os.system("convert -delay 1 -dispose Background +page " + str(output_dir)
          + "/*.png -loop 0 " + str(output_dir) + "/animation.gif")
```

Esto requiere tener ImageMagick instalado. La versión de abajo hace lo mismo **sin dependencias
externas**, usando `matplotlib.animation` con el escritor Pillow, y produce directamente un GIF.
Se conserva la estructura del programa original: `compute_next_event` calcula el próximo evento,
`compute_new_velocities` aplica la regla de colisión, y el bucle principal avanza el sistema en
pasos de tiempo **fijos** `dt` (para tener fotogramas equiespaciados), resolviendo los eventos que
caigan dentro de cada paso.

In [ ]:
# =====================================================================
# ANIMACIÓN — dinámica por eventos (versión sin ImageMagick)
# =====================================================================
import matplotlib.animation as animation
from matplotlib.animation import PillowWriter

def min_arg(l):
    """(28-29) Devuelve (valor_mínimo, índice_del_mínimo) de la lista l."""
    return min(zip(l, range(len(l))))

def compute_next_event(pos, vel, sigma, singles, pairs):
    """(31-34) Calcula el próximo evento: devuelve (tiempo, índice del evento)."""
    wall_times = [wall_time(pos[k][l], vel[k][l], sigma) for k, l in singles]
    pair_times = [pair_time(pos[k], vel[k], pos[l], vel[l], sigma) for k, l in pairs]
    return min_arg(wall_times + pair_times)

def compute_new_velocities(pos, vel, next_event_arg, singles, pairs):
    """(36-49) Aplica la regla de colisión que corresponda."""
    if next_event_arg < len(singles):
        # --- colisión con la pared: se invierte la componente perpendicular
        collision_disk, direction = singles[next_event_arg]
        vel[collision_disk][direction] *= -1.0
    else:
        # --- colisión entre dos discos: intercambio de la componente longitudinal
        a, b = pairs[next_event_arg - len(singles)]
        del_x  = [pos[b][0]-pos[a][0], pos[b][1]-pos[a][1]]
        abs_x  = math.sqrt(del_x[0]**2 + del_x[1]**2)
        e_perp = [c/abs_x for c in del_x]
        del_v  = [vel[b][0]-vel[a][0], vel[b][1]-vel[a][1]]
        scal   = del_v[0]*e_perp[0] + del_v[1]*e_perp[1]
        for k in range(2):
            vel[a][k] += e_perp[k]*scal
            vel[b][k] -= e_perp[k]*scal


def animar_discos(pos0, vel0, sigma, n_frames=200, dt=0.02,
                  nombre="animacion_discos.gif", fps=25, estelas=True):
    """
    Genera un GIF de la dinámica por eventos.
    `dt` es el intervalo entre FOTOGRAMAS (no un paso de integración: dentro de cada
    intervalo los eventos se resuelven exactamente).
    """
    pos = [list(p) for p in pos0]
    vel = [list(v) for v in vel0]
    N = len(pos)
    singles = [(k, l) for k in range(N) for l in range(2)]
    pairs   = [(k, l) for k in range(N) for l in range(k+1, N)]
    colores_d = plt.cm.tab10(np.linspace(0, 1, 10))

    # --- se precomputan todas las configuraciones de los fotogramas ---
    trayectoria = []
    t = 0.0
    next_event, next_event_arg = compute_next_event(pos, vel, sigma, singles, pairs)
    for frame in range(n_frames):
        next_t = t + dt
        # se resuelven TODOS los eventos que ocurran antes del próximo fotograma
        while t + next_event <= next_t:
            for k, l in singles:
                pos[k][l] += vel[k][l]*next_event
            t += next_event
            compute_new_velocities(pos, vel, next_event_arg, singles, pairs)
            next_event, next_event_arg = compute_next_event(pos, vel, sigma, singles, pairs)
        # avance balístico hasta el instante del fotograma
        remain = next_t - t
        for k, l in singles:
            pos[k][l] += vel[k][l]*remain
        next_event -= remain
        t = next_t
        trayectoria.append(([list(p) for p in pos], [list(v) for v in vel], t))

    # --- render ---------------------------------------------------------
    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xticks([0, 0.5, 1]); ax.set_yticks([0, 0.5, 1]); ax.grid(False)
    circulos = [patches.Circle((0, 0), sigma, facecolor=colores_d[k % 10],
                               edgecolor="k", lw=0.8, alpha=0.9) for k in range(N)]
    for c in circulos: ax.add_patch(c)
    flechas = [None]*N
    titulo = ax.set_title("")

    def update(i):
        p, v, tt = trayectoria[i]
        for k in range(N):
            circulos[k].center = (p[k][0], p[k][1])
            if flechas[k] is not None:
                flechas[k].remove()
            flechas[k] = ax.arrow(p[k][0], p[k][1], v[k][0]*0.18, v[k][1]*0.18,
                                  head_width=0.028, head_length=0.035,
                                  fc="k", ec="k", lw=0.7)
        titulo.set_text("Dinámica molecular por eventos   $t = %.2f$" % tt)
        return circulos + [f for f in flechas if f is not None] + [titulo]

    anim = animation.FuncAnimation(fig, update, frames=len(trayectoria), blit=False)
    anim.save(nombre, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return nombre


# --- Generación de la animación ---------------------------------------
random.seed(2026)
sigma_anim = 0.15
pos_anim = [[0.25, 0.25], [0.75, 0.25], [0.25, 0.75], [0.75, 0.75]]
vel_anim = [[0.21, 0.12], [0.71, 0.18], [-0.23, -0.79], [0.78, 0.1177]]

archivo = animar_discos(pos_anim, vel_anim, sigma_anim,
                        n_frames=150 if FAST_MODE else 400, dt=0.02)
print("Animación guardada en:", archivo)

from IPython.display import Image, display
display(Image(filename=archivo))

### Versión con ImageMagick (la del enunciado)

Si prefieres reproducir literalmente el programa de la diapositiva 12, guarda cada fotograma como
PNG y llama a `convert`. Requiere ImageMagick instalado (`sudo apt install imagemagick`):

```python
import os
output_dir = "event_disks_box_movie"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def snapshot(t, pos, vel, colors, arrow_scale=0.2):
    global img
    plt.cla()
    plt.axis([0, 1, 0, 1])
    plt.setp(plt.gca(), xticks=[0, 1], yticks=[0, 1])
    for (x, y), (dx, dy), c in zip(pos, vel, colors):
        dx *= arrow_scale
        dy *= arrow_scale
        circle = plt.Circle((x, y), radius=sigma, fc=c)
        plt.gca().add_patch(circle)
        plt.arrow(x, y, dx, dy, fc="k", ec="k", head_width=0.05, head_length=0.05)
    plt.text(0.5, 1.03, 't = %.2f' % t, ha='center')
    plt.savefig(os.path.join(output_dir, '%04i.png' % img))
    img += 1

os.system("convert -delay 1 -dispose Background +page " + str(output_dir)
          + "/*.png -loop 0 " + str(output_dir) + "/animation.gif")
```

Es exactamente lo mismo que hace `animar_discos`, sólo que delegando el ensamblado del GIF a un
programa externo.

---
# BONO EXTRA — Interacción dipolar magnética

## Marco teórico

Se dota a cada disco de un momento magnético $\boldsymbol{\mu}_i$ de módulo fijo $\mu$ y
orientación libre $\theta_i$ en el plano. La energía de interacción dipolo–dipolo entre dos
momentos separados por $\mathbf{r}_{ij}$ es

$$U_{ij} = \frac{\mu_0}{4\pi r_{ij}^3}
\Big[\,\boldsymbol{\mu}_i\cdot\boldsymbol{\mu}_j
- 3(\boldsymbol{\mu}_i\cdot\hat{\mathbf{r}}_{ij})(\boldsymbol{\mu}_j\cdot\hat{\mathbf{r}}_{ij})\Big].$$

Características que la hacen especial:

- Es **anisotrópica**: la energía depende de la orientación relativa respecto a la línea que une
  los dipolos. Dos dipolos alineados **cabeza con cola** se atraen (energía $-2\mu^2/r^3$),
  mientras que dos dipolos paralelos **lado a lado** se repelen ($+\mu^2/r^3$).
- Es de **largo alcance** ($\sim r^{-3}$).
- Esa anisotropía es la responsable de que los ferrofluidos formen **cadenas y anillos** al bajar
  la temperatura o subir la concentración, y de las estructuras en espiga bajo campo externo.

**¿Por qué no se usa dinámica por eventos aquí?** Porque el algoritmo de Alder–Wainwright supone
movimiento **balístico entre colisiones**, lo cual sólo vale si la fuerza es nula fuera del
contacto. Con la interacción dipolar hay fuerzas continuas y las trayectorias dejan de ser rectas.
Las alternativas correctas son (i) dinámica molecular con integrador de paso finito
(Verlet) más un tratamiento especial del núcleo duro, o (ii) **Monte Carlo de Metropolis**, que es
lo que se implementa abajo por ser más simple y exacto en equilibrio.

Ahora el sistema **no** es microcanónico: al haber energía potencial, el peso estadístico es el de
Boltzmann, $\pi(\mathbf{x}) \propto e^{-\beta U(\mathbf{x})}$, y la aceptación de Metropolis deja
de ser trivial:

$$P_{\rm acc} = \min\left(1,\; e^{-\beta \Delta U}\right).$$

El parámetro de control es el **acoplamiento dipolar reducido**
$\lambda = \mu_0\mu^2 / (4\pi (2\sigma)^3 k_B T)$: para $\lambda \lesssim 1$ el sistema se comporta
como un gas casi ideal; para $\lambda \gtrsim 3$ aparecen las cadenas características del
ferrofluido.

In [ ]:
# =====================================================================
# BONO — Monte Carlo de Metropolis con interacción dipolar magnética
# =====================================================================

def energia_dipolar(pos, ang, lam, sigma, k=None):
    """
    Energía dipolar total (o sólo la del disco k, si se especifica), en unidades
    reducidas: U/kT = lam * (2 sigma)^3 * [mu_i.mu_j - 3(mu_i.r)(mu_j.r)] / r^3
    con |mu| = 1. `lam` es el acoplamiento dipolar reducido.
    """
    N = len(pos)
    pref = lam * (2*sigma)**3
    indices = range(N) if k is None else [k]
    U = 0.0
    for i in indices:
        mix, miy = math.cos(ang[i]), math.sin(ang[i])
        for j in range(N):
            if j == i or (k is None and j <= i):
                continue
            dx = pos[j][0]-pos[i][0]; dy = pos[j][1]-pos[i][1]
            r2 = dx*dx + dy*dy
            r  = math.sqrt(r2); r3 = r2*r
            ex, ey = dx/r, dy/r
            mjx, mjy = math.cos(ang[j]), math.sin(ang[j])
            mi_mj = mix*mjx + miy*mjy
            mi_e  = mix*ex + miy*ey
            mj_e  = mjx*ex + mjy*ey
            U += pref * (mi_mj - 3.0*mi_e*mj_e) / r3
    return U


def mc_dipolar(N, sigma, lam, n_steps, delta=0.06, dtheta=0.6, semilla=0,
               guardar_cada=None):
    """
    Cadena de Markov (Metropolis) para N discos rígidos con momento magnético.
    Cada paso propone, con igual probabilidad, una traslación o una rotación.
    Devuelve las instantáneas guardadas y la energía media.
    """
    random.seed(semilla)
    pos = [list(p) for p in direct_disks_box_sq(N, sigma)]
    ang = [random.uniform(0, 2*math.pi) for _ in range(N)]
    cuatro_ssq = 4.0*sigma**2
    instantaneas = []
    energias = []
    n_acc = 0
    if guardar_cada is None:
        guardar_cada = max(1, n_steps//200)

    for paso in range(n_steps):
        k = random.randint(0, N-1)
        U_old = energia_dipolar(pos, ang, lam, sigma, k=k)

        if random.random() < 0.5:
            # --- movimiento de TRASLACIÓN ---
            b0 = pos[k][0] + random.uniform(-delta, delta)
            b1 = pos[k][1] + random.uniform(-delta, delta)
            if (b0 < sigma or b1 < sigma or b0 > 1-sigma or b1 > 1-sigma):
                legal = False
            else:
                legal = all((b0-pos[j][0])**2 + (b1-pos[j][1])**2 >= cuatro_ssq
                            for j in range(N) if j != k)
            if legal:
                viejo = pos[k]; pos[k] = [b0, b1]
                dU = energia_dipolar(pos, ang, lam, sigma, k=k) - U_old
                if dU <= 0 or random.random() < math.exp(-dU):
                    n_acc += 1           # aceptado
                else:
                    pos[k] = viejo       # rechazado: se revierte
        else:
            # --- movimiento de ROTACIÓN del momento magnético ---
            viejo = ang[k]
            ang[k] = (ang[k] + random.uniform(-dtheta, dtheta)) % (2*math.pi)
            dU = energia_dipolar(pos, ang, lam, sigma, k=k) - U_old
            if dU <= 0 or random.random() < math.exp(-dU):
                n_acc += 1
            else:
                ang[k] = viejo

        if paso % guardar_cada == 0:
            instantaneas.append(([list(p) for p in pos], list(ang)))
            energias.append(energia_dipolar(pos, ang, lam, sigma))

    return instantaneas, np.array(energias), n_acc/n_steps


def dibujar_dipolos(pos, ang, sigma, ax, titulo=""):
    for (x, y), th in zip(pos, ang):
        ax.add_patch(patches.Circle((x, y), sigma, facecolor="#c6dbef",
                                    edgecolor="#2171b5", lw=1.0))
        ax.arrow(x - 0.7*sigma*math.cos(th), y - 0.7*sigma*math.sin(th),
                 1.4*sigma*math.cos(th), 1.4*sigma*math.sin(th),
                 head_width=0.6*sigma, head_length=0.6*sigma,
                 fc="crimson", ec="crimson", lw=1.0, length_includes_head=True)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title(titulo, fontsize=9)

print("Monte Carlo dipolar listo.")

In [ ]:
# =====================================================================
# BONO — Efecto del acoplamiento dipolar y de la concentración
# =====================================================================
n_mc = 60_000 if FAST_MODE else 400_000
casos = [
    (12, 0.055, 0.0, "Sin interacción\n$\\lambda=0$ (gas ideal de discos)"),
    (12, 0.055, 2.0, "Acoplamiento débil\n$\\lambda=2$"),
    (12, 0.055, 6.0, "Acoplamiento fuerte\n$\\lambda=6$ (ferrofluido: cadenas)"),
    (24, 0.055, 6.0, "Alta concentración\n$N=24$, $\\lambda=6$"),
]

fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (Nd, sd, lam, tit) in zip(axes, casos):
    snaps, ener, acc = mc_dipolar(Nd, sd, lam, n_mc, semilla=13)
    pos_f, ang_f = snaps[-1]
    dibujar_dipolos(pos_f, ang_f, sd, ax,
                    "%s\n$\\eta=%.2f$, acept.=%.0f%%, $U/N=%.2f$"
                    % (tit, Nd*math.pi*sd**2, 100*acc, ener[-1]/Nd))
plt.suptitle("Bono: sistema de discos rígidos con momento magnético "
             "(Monte Carlo de Metropolis)", y=1.06)
plt.tight_layout()
plt.savefig("fig12_dipolar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# =====================================================================
# BONO — Animación del ferrofluido y curva de energía
# =====================================================================
snaps, ener, acc = mc_dipolar(16, 0.055, 6.0, n_mc, semilla=21, guardar_cada=max(1, n_mc//180))

fig, ax = plt.subplots(figsize=(5.0, 5.0))
def update_dip(i):
    ax.clear()
    p, a = snaps[i]
    dibujar_dipolos(p, a, 0.055, ax, "Ferrofluido 2D, $\\lambda=6$   (paso MC %d)"
                    % (i*max(1, n_mc//180)))
    return []
anim = animation.FuncAnimation(fig, update_dip, frames=len(snaps), blit=False)
anim.save("animacion_ferrofluido.gif", writer=PillowWriter(fps=15))
plt.close(fig)
print("Animación guardada en: animacion_ferrofluido.gif")
display(Image(filename="animacion_ferrofluido.gif"))

# Termalización de la energía
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(np.arange(len(ener))*max(1, n_mc//180), ener/16, lw=1.2, color="tab:purple")
ax.set_xlabel("paso de Monte Carlo"); ax.set_ylabel("$U/N$  [unidades de $k_BT$]")
ax.set_title("Termalización de la energía dipolar ($\\lambda=6$, $N=16$)")
plt.tight_layout(); plt.savefig("fig13_energia_dipolar.png", dpi=150)
plt.show()

---
# Resumen de resultados y conclusiones

Puntos que conviene destacar en el informe:

1. **La equiprobabilidad se verifica numéricamente** con los tres métodos: $\hat p_a \simeq
   \hat p_b \simeq \hat p_c$ dentro de las barras de error, tanto para $N=4$ como para $N=8$, y los
   valores absolutos coinciden entre muestreo directo, cadena de Markov y dinámica por eventos.

2. **La precisión la controla el número de aciertos, no $n_{\rm runs}$**: el error relativo es
   $\sim 1/\sqrt{\text{hits}}$. Con $p\sim10^{-4}$ hacen falta $\gtrsim 10^6$ corridas para bajar
   del 10 % de error.

3. **El tamaño de las cajitas es un compromiso sesgo–varianza**: demasiado grandes producen
   solapamiento de ventanas y sesgo sistemático; demasiado pequeñas anulan la estadística porque
   $p\propto \delta^{2N}$.

4. **El muestreo directo da muestras independientes; la cadena de Markov no.** Las fluctuaciones
   de MCMC exceden la predicción de Poisson en un factor $\sqrt{2\tau_{\rm int}}$. A cambio, MCMC
   sigue funcionando a densidades donde la tasa de aceptación del muestreo directo colapsa.

5. **La dinámica por eventos es exacta** (energía conservada a precisión de máquina) pero exige
   muestrear a **tiempos equiespaciados**; muestrear en los instantes de colisión introduce un
   sesgo sistemático.

6. **El histograma de $x$ no es plano, y eso no contradice la equiprobabilidad.** Es la marginal de
   una distribución uniforme sobre una región de forma no trivial; el perfil refleja el volumen
   excluido y la fuerza entrópica de depleción. Se aplana al bajar la densidad y es exactamente
   plano con contornos periódicos.

---

# Bibliografía sugerida

1. W. Krauth, *Statistical Mechanics: Algorithms and Computations*, Oxford University Press (2006).
   — Referencia directa de estos algoritmos; los programas del enunciado provienen de este texto y
   del curso asociado.
2. B. J. Alder y T. E. Wainwright, *Phase Transition for a Hard Sphere System*,
   J. Chem. Phys. **27**, 1208 (1957). — Artículo fundacional de la dinámica molecular por eventos.
3. B. J. Alder y T. E. Wainwright, *Studies in Molecular Dynamics. I. General Method*,
   J. Chem. Phys. **31**, 459 (1959).
4. N. Metropolis, A. W. Rosenbluth, M. N. Rosenbluth, A. H. Teller y E. Teller,
   *Equation of State Calculations by Fast Computing Machines*, J. Chem. Phys. **21**, 1087 (1953).
   — Origen del muestreo de Metropolis, aplicado justamente a discos rígidos.
5. D. Frenkel y B. Smit, *Understanding Molecular Simulation: From Algorithms to Applications*,
   2ª ed., Academic Press (2002).
6. M. P. Allen y D. J. Tildesley, *Computer Simulation of Liquids*, 2ª ed., Oxford (2017).
7. S. Asakura y F. Oosawa, *On Interaction between Two Bodies Immersed in a Solution of
   Macromolecules*, J. Chem. Phys. **22**, 1255 (1954). — Fuerza de depleción.
8. E. M. Kirkpatrick y otros / R. E. Rosensweig, *Ferrohydrodynamics*, Cambridge (1985).
   — Interacción dipolar y ferrofluidos.
9. K. Huang, *Statistical Mechanics*, 2ª ed., Wiley (1987). — Postulado de equiprobabilidad a
   priori y colectivo microcanónico.

---

# Agradecimientos

_(Recuerda incluir aquí los enlaces a los hilos de conversación con IA que hayas usado, tal como
exige la nota del enunciado.)_